This notebook documents the complete data collection and preprocessing pipeline for the AgriGuard project.

The dataset was collected from two main sources:

1. **Satellite Data Source:**  
   Google Earth Engine (GEE) using **Sentinel-2 Surface Reflectance Harmonized**
   to extract vegetation and water indices:
   - NDVI
   - NDWI
   - EVI

   Data was extracted for **True 1km × 1km agricultural grids**
   covering **Kafr El-Sheikh Governorate, Egypt**
   over the period **2020–2024**
   with a **15-day temporal interval**.

2. **Climate Data Source:**  
   **NASA POWER API**
   for daily climate variables including:
   - Temperature
   - Rainfall
   - Humidity
   - Wind Speed

This notebook also includes preprocessing, cleaning, validation,
and temporal alignment of the extracted datasets.

In [ ]:
# ==========================================================
# Data Handling
# ==========================================================
import os
import warnings

import joblib
import numpy as np
import pandas as pd

# ==========================================================
# Visualization
# ==========================================================
import matplotlib.pyplot as plt
import seaborn as sns
import folium

from branca.colormap import LinearColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================================
# Machine Learning
# ==========================================================
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

# ==========================================================
# Imbalanced Learning
# ==========================================================
from imblearn.combine import SMOTETomek

# ==========================================================
# Models
# ==========================================================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
import xgboost as xgb

# ==========================================================
# Explainability
# ==========================================================
import shap

# ==========================================================
# Warnings
# ==========================================================
warnings.filterwarnings("ignore")

# **Satellite**

# **Dataset Comprehensive Audit & Integrity Check**

This cell performs a full diagnostic on the new UTM-based 1km x 1km grids and their corresponding 5-year time series.

In [ ]:
# 1. Load the New Dataset Files
# Make sure to upload 'Kafr_ElSheikh_True1km_Grids_List.csv'
# and the new 'Kafr_ElSheikh_True1km_TimeSeries_2020_2024.csv'
grids_file = 'Kafr_ElSheikh_True1km_Grids_List.csv'
ts_file = 'Kafr_ElSheikh_True1km_TimeSeries_2020_2024.csv'

try:
    grids_df = pd.read_csv(grids_file)
    ts_df = pd.read_csv(ts_file)
    print(" Files loaded successfully!")
except FileNotFoundError:
    print(" Error: Please make sure you have uploaded the correct files to Colab.")

# --- SECTION 1: Spatial Inventory (True 1km Grids) ---
print("\n--- SECTION 1: SPATIAL DATA AUDIT ---")
total_unique_grids = grids_df['Grid_ID'].nunique()
print(f"Total Verified Agricultural Grids: {total_unique_grids}")
print(f"Latitude Range:  {grids_df['lat'].min():.4f} to {grids_df['lat'].max():.4f}")
print(f"Longitude Range: {grids_df['lon'].min():.4f} to {grids_df['lon'].max():.4f}")

# --- SECTION 2: Temporal Overview (Time-Series) ---
print("\n--- SECTION 2: TEMPORAL DATA AUDIT ---")
print(f"Total Rows in Dataset: {len(ts_df):,}")
print(f"Total Unique Dates:    {ts_df['date'].nunique()}")
print(f"Start Date:            {ts_df['date'].min()}")
print(f"End Date:              {ts_df['date'].max()}")

# Verify if every grid has exactly 120 steps
avg_records = ts_df['Grid_ID'].value_counts().mean()
print(f"Average Records per Grid: {avg_records:.1f} (Target: 120.0)")

# --- SECTION 3: Missing Values & EVI Stability ---
print("\n--- SECTION 3: DATA QUALITY & EVI STABILITY ---")
null_report = pd.DataFrame({
    'Missing Values': ts_df.isnull().sum(),
    'Percentage (%)': (ts_df.isnull().sum() / len(ts_df)) * 100
})
print(null_report)

# Now checking if EVI is within scientific range (-1 to 1)
print("\n--- SECTION 4: SCIENTIFIC INDICES SUMMARY ---")
stats = ts_df[['NDVI', 'NDWI', 'EVI']].describe().loc[['min', 'max', 'mean']]
print(stats)

# --- SECTION 5: Data Preview ---
print("\n--- SECTION 5: DATA PREVIEW (NEW GRID IDs) ---")
print(ts_df.head())

# **Raw Data Visualization (Pre-Cleaning)**

Plotting NDVI, NDWI, and EVI exactly as they are in the file to observe outliers and gaps before any processing.

In [ ]:
# 1. Ensure date format is correct
ts_df['date'] = pd.to_datetime(ts_df['date'])

# 2. Select the specific grid for analysis
# You can change this ID to any other grid (e.g., "G_332_3486" has big outliers)
sample_grid_id = "G_294_3432"
grid_data = ts_df[ts_df['Grid_ID'] == sample_grid_id].sort_values('date')

# --- PLOT 1: RAW NDVI ---
plt.figure(figsize=(15, 5))
plt.plot(grid_data['date'], grid_data['NDVI'], marker='o', linestyle='-', color='#2ca02c', markersize=4, label='Raw NDVI')
plt.title(f'RAW NDVI Time Series - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('NDVI Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# --- PLOT 2: RAW NDWI ---
plt.figure(figsize=(15, 5))
plt.plot(grid_data['date'], grid_data['NDWI'], marker='s', linestyle='-', color='#1f77b4', markersize=4, label='Raw NDWI')
plt.title(f'RAW NDWI Time Series - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('NDWI Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# --- PLOT 3: RAW EVI  ---
plt.figure(figsize=(15, 5))
plt.plot(grid_data['date'], grid_data['EVI'], marker='^', linestyle='-', color='#9467bd', markersize=4, label='Raw EVI')
plt.title(f'RAW EVI Time Series (With Outliers) - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('EVI Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(f" RAW Visualization complete for Grid: {sample_grid_id}")

# **Befor Preprocessing**

### **Satellite Indices Validation & Diagnostic Tests**

**Raw Data Validation & Diagnostic Analysis**

This section performs a full validation and diagnostic analysis for the extracted satellite vegetation and water indices (NDVI, NDWI, EVI) over the True 1km agricultural grids in Kafr El-Sheikh.

**Important Notes**

- Missing values have NOT been handled yet.  
- EVI outliers have NOT been cleaned yet.  
- This audit is performed on the RAW extracted dataset.  
- The goal is to evaluate data quality, scientific ranges, temporal consistency, seasonal behavior, and inter-index relationships BEFORE preprocessing.

In [ ]:
# -----------------------------------------------------------------------------
# 1. LOAD DATA
# -----------------------------------------------------------------------------
file_path = 'Kafr_ElSheikh_True1km_TimeSeries_2020_2024.csv'

df = pd.read_csv(file_path)
print(" File loaded successfully!")
print(f"Dataset shape: {df.shape}")

# Convert date column
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Basic check
print("\n--- BASIC DATA PREVIEW ---")
print(df.head())

# -----------------------------------------------------------------------------
# 2. GENERAL DATASET OVERVIEW
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 1: GENERAL DATASET OVERVIEW")
print("="*80)

print(f"Total Rows: {len(df):,}")
print(f"Total Unique Grids: {df['Grid_ID'].nunique():,}")
print(f"Total Unique Dates: {df['date'].nunique():,}")
print(f"Start Date: {df['date'].min()}")
print(f"End Date:   {df['date'].max()}")

avg_records = df['Grid_ID'].value_counts().mean()
print(f"Average Records per Grid: {avg_records:.2f}")

# -----------------------------------------------------------------------------
# 3. MISSING VALUES REPORT
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 2: MISSING VALUES REPORT")
print("="*80)

missing_report = pd.DataFrame({
    'Missing Values': df[['NDVI', 'NDWI', 'EVI']].isnull().sum(),
    'Missing Percentage (%)': (df[['NDVI', 'NDWI', 'EVI']].isnull().sum() / len(df)) * 100
})
print(missing_report)

print("\nNOTE:")
print("- Missing values are still present and have NOT been interpolated yet.")
print("- This is expected at this stage because preprocessing has not started.")

# -----------------------------------------------------------------------------
# 4. SCIENTIFIC RANGE VALIDATION
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 3: SCIENTIFIC RANGE VALIDATION")
print("="*80)

for col in ['NDVI', 'NDWI', 'EVI']:
    valid_mask = df[col].notna()
    out_of_range = ((df[col] < -1) | (df[col] > 1)) & valid_mask
    count_bad = out_of_range.sum()
    total_non_null = valid_mask.sum()
    pct_bad = (count_bad / total_non_null) * 100 if total_non_null > 0 else 0

    print(f"\n{col}:")
    print(f"  Non-missing values: {total_non_null:,}")
    print(f"  Out-of-range values [-1, 1]: {count_bad:,}")
    print(f"  Out-of-range percentage: {pct_bad:.4f}%")

print("\nNOTE:")
print("- NDVI and NDWI are expected to remain mostly within [-1, 1].")
print("- EVI may contain outliers at this stage because its outlier problem has not been cleaned yet.")

# -----------------------------------------------------------------------------
# 5. DESCRIPTIVE STATISTICS
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 4: DESCRIPTIVE STATISTICS")
print("="*80)

stats = df[['NDVI', 'NDWI', 'EVI']].describe().T
print(stats[['min', '25%', '50%', 'mean', '75%', 'max', 'std']])

print("\nINTERPRETATION NOTE:")
print("- Median (50%) is very important, especially for EVI.")
print("- Mean may be misleading if there are extreme outliers.")

# -----------------------------------------------------------------------------
# 6. CORRELATION ANALYSIS
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 5: CORRELATION ANALYSIS")
print("="*80)

corr_matrix = df[['NDVI', 'NDWI', 'EVI']].corr()
print(corr_matrix)

# Plot correlation matrix manually using matplotlib
plt.figure(figsize=(6, 5))
plt.imshow(corr_matrix, interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title("Correlation Matrix: NDVI / NDWI / EVI")
for i in range(len(corr_matrix.index)):
    for j in range(len(corr_matrix.columns)):
        plt.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha='center', va='center')
plt.tight_layout()
plt.show()

print("\nINTERPRETATION NOTE:")
print("- NDVI and EVI are expected to have strong positive correlation.")
print("- NDWI may also correlate positively but usually less strongly.")

# -----------------------------------------------------------------------------
# 7. MONTHLY SEASONALITY CHECK
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 6: MONTHLY SEASONALITY CHECK")
print("="*80)

df['month'] = df['date'].dt.month

monthly_avg = df.groupby('month')[['NDVI', 'NDWI', 'EVI']].mean()

print("Monthly Average Values:")
print(monthly_avg)

plt.figure(figsize=(12, 6))
plt.plot(monthly_avg.index, monthly_avg['NDVI'], marker='o', label='NDVI')
plt.plot(monthly_avg.index, monthly_avg['NDWI'], marker='s', label='NDWI')
plt.plot(monthly_avg.index, monthly_avg['EVI'], marker='^', label='EVI')
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Average Index Value")
plt.title("Monthly Seasonality Pattern of NDVI, NDWI, and EVI")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

print("\nINTERPRETATION NOTE:")
print("- A meaningful seasonal pattern should appear across months.")
print("- Vegetation indices should rise and fall according to agricultural cycles.")

# -----------------------------------------------------------------------------
# 8. SAMPLE GRID TEMPORAL CONSISTENCY CHECK
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 7: SAMPLE GRID TEMPORAL CONSISTENCY CHECK")
print("="*80)

# Select sample grids
sample_grids = df['Grid_ID'].drop_duplicates().sample(5, random_state=42).tolist()
print("Sample Grids Selected:")
print(sample_grids)

for gid in sample_grids:
    sub = df[df['Grid_ID'] == gid].sort_values('date')

    plt.figure(figsize=(14, 4))
    plt.plot(sub['date'], sub['NDVI'], marker='o', label='NDVI')
    plt.plot(sub['date'], sub['NDWI'], marker='s', label='NDWI')
    plt.plot(sub['date'], sub['EVI'], marker='^', label='EVI')
    plt.title(f"Raw Time Series for Grid: {gid}")
    plt.xlabel("Date")
    plt.ylabel("Index Value")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

print("\nINTERPRETATION NOTE:")
print("- The time series should show seasonal cycles, not random chaos.")
print("- EVI spikes, if present, are expected at this raw stage before cleaning.")

# -----------------------------------------------------------------------------
# 9. RAW EVI OUTLIER INSPECTION
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 8: RAW EVI OUTLIER INSPECTION")
print("="*80)

evi_outliers = df[(df['EVI'] < -1) | (df['EVI'] > 1)]

print(f"Total Raw EVI Outliers: {len(evi_outliers):,}")

if len(evi_outliers) > 0:
    print("\nSample of EVI Outliers:")
    print(evi_outliers[['Grid_ID', 'date', 'EVI']].head(10))
else:
    print("No EVI outliers detected outside [-1, 1].")

print("\nNOTE:")
print("- These outliers have NOT been removed yet.")
print("- This section only detects and quantifies them.")

# -----------------------------------------------------------------------------
# 10. GRID COMPLETENESS CHECK
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 9: GRID COMPLETENESS CHECK")
print("="*80)

grid_counts = df['Grid_ID'].value_counts()

print("Records per Grid Summary:")
print(grid_counts.describe())

expected_steps = 120
incomplete_grids = grid_counts[grid_counts != expected_steps]

print(f"\nExpected Records per Grid: {expected_steps}")
print(f"Grids NOT matching expected count: {len(incomplete_grids)}")

if len(incomplete_grids) > 0:
    print("\nSample incomplete grids:")
    print(incomplete_grids.head(10))
else:
    print(" All grids have the expected number of temporal records.")

# -----------------------------------------------------------------------------
# 11. FINAL VALIDATION SUMMARY
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 10: FINAL VALIDATION SUMMARY")
print("="*80)

print("1. The dataset structure has been checked.")
print("2. Missing values were quantified but NOT treated yet.")
print("3. Scientific ranges were validated.")
print("4. Correlation between NDVI, NDWI, and EVI was examined.")
print("5. Monthly seasonality was checked.")
print("6. Sample time-series consistency across multiple grids was visualized.")
print("7. Raw EVI outliers were detected but NOT cleaned yet.")
print("8. Grid temporal completeness was verified.")

print("\nFINAL NOTE:")
print("This is a RAW diagnostic audit only.")
print("Preprocessing steps such as missing-value interpolation and EVI outlier handling")
print("must be performed before using the dataset for machine learning.")

# **Time Series Preprocessing: Missing Values Interpolation & EVI Outlier Handling**




**Data Cleaning & Preprocessing**

This step focuses on cleaning the raw Kafr El-Sheikh True 1km time-series dataset to prepare it for machine learning.

**Cleaning Steps**

1. Convert EVI outliers to NaN  
2. Interpolate missing values within each Grid_ID over time  
3. Apply forward and backward fill for edge cases  
4. Save the final cleaned dataset for further analysis

In [ ]:
import os

print(os.getcwd())
print(os.path.exists('Kafr_ElSheikh_True1km_TimeSeries_2020_2024.csv'))

In [ ]:
# -----------------------------------------------------------------------------
# 1. Load Raw Dataset
# -----------------------------------------------------------------------------
file_path = 'Kafr_ElSheikh_True1km_TimeSeries_2020_2024.csv'
df = pd.read_csv(file_path)

print("Raw dataset loaded successfully!")
print("Original shape:", df.shape)

# -----------------------------------------------------------------------------
# 2. Prepare Date & Sorting
# -----------------------------------------------------------------------------
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values(['Grid_ID', 'date']).reset_index(drop=True)

print("\nDate converted and dataset sorted by Grid_ID and date.")

# -----------------------------------------------------------------------------
# 3. Initial Missing Values Report
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 1: Initial Missing Values Report")
print("="*80)

print(df[['NDVI', 'NDWI', 'EVI']].isnull().sum())

# -----------------------------------------------------------------------------
# 4. Convert EVI Outliers to NaN
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 2: EVI Outlier Handling")
print("="*80)

evi_outliers_before = ((df['EVI'] < -1) | (df['EVI'] > 1)).sum()
print(f"Raw EVI outliers outside [-1, 1]: {evi_outliers_before:,}")

df.loc[(df['EVI'] < -1) | (df['EVI'] > 1), 'EVI'] = np.nan

print("EVI outliers converted to NaN successfully.")

# -----------------------------------------------------------------------------
# 5. Missing Values After Outlier Removal
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 3: Missing Values After EVI Outlier Removal")
print("="*80)

print(df[['NDVI', 'NDWI', 'EVI']].isnull().sum())

# -----------------------------------------------------------------------------
# 6. Interpolation Within Each Grid_ID
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 4: Linear Interpolation")
print("="*80)

for col in ['NDVI', 'NDWI', 'EVI']:
    df[col] = (
        df.groupby('Grid_ID')[col]
        .transform(lambda s: s.interpolate(method='linear', limit_direction='both'))
    )

print("Linear interpolation completed.")

# -----------------------------------------------------------------------------
# 7. Forward Fill / Backward Fill
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 5: Forward Fill / Backward Fill")
print("="*80)

for col in ['NDVI', 'NDWI', 'EVI']:
    df[col] = (
        df.groupby('Grid_ID')[col]
        .transform(lambda s: s.ffill().bfill())
    )

print("Forward fill and backward fill completed.")

# -----------------------------------------------------------------------------
# 8. Final Validation
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 6: Final Validation")
print("="*80)

print("Remaining Missing Values:")
print(df[['NDVI', 'NDWI', 'EVI']].isnull().sum())

print("\nRemaining Out-of-Range Values:")
for col in ['NDVI', 'NDWI', 'EVI']:
    outliers = ((df[col] < -1) | (df[col] > 1)).sum()
    print(f"{col}: {outliers}")

# -----------------------------------------------------------------------------
# 9. Save Cleaned Dataset
# -----------------------------------------------------------------------------
output_file = 'Kafr_ElSheikh_True1km_TimeSeries_2020_2024_Cleaned.csv'
df.to_csv(output_file, index=False)

print("\n" + "="*80)
print("SECTION 7: Export")
print("="*80)
print(f"Cleaned dataset saved successfully as: {output_file}")

# **After Preprocessing**

### **Cleaned Satellite Indices Validation & Final Diagnostic Tests**

**Post-Cleaning Data Validation & Diagnostic Analysis**

This section performs a full validation and diagnostic analysis for the satellite vegetation and water indices (NDVI, NDWI, EVI) over the True 1km agricultural grids in Kafr El-Sheikh after preprocessing.

**Important Notes**

- Missing values have been fully handled using interpolation  
- EVI outliers have been cleaned and validated  
- This audit is performed on the final cleaned dataset  

 **Objectives**

- Confirm overall data quality  
- Validate scientific ranges of the indices  
- Ensure temporal consistency across grids  
- Verify seasonal patterns  
- Examine relationships between indices before machine learning

In [ ]:
# -----------------------------------------------------------------------------
# 1. LOAD DATA
# -----------------------------------------------------------------------------
file_path = 'Kafr_ElSheikh_True1km_TimeSeries_2020_2024_Cleaned.csv'

df = pd.read_csv(file_path)
print("File loaded successfully!")
print(f"Dataset shape: {df.shape}")

# Convert date column
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Basic check
print("\n--- BASIC DATA PREVIEW ---")
print(df.head())

# -----------------------------------------------------------------------------
# 2. GENERAL DATASET OVERVIEW
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 1: GENERAL DATASET OVERVIEW")
print("="*80)

print(f"Total Rows: {len(df):,}")
print(f"Total Unique Grids: {df['Grid_ID'].nunique():,}")
print(f"Total Unique Dates: {df['date'].nunique():,}")
print(f"Start Date: {df['date'].min()}")
print(f"End Date:   {df['date'].max()}")

avg_records = df['Grid_ID'].value_counts().mean()
print(f"Average Records per Grid: {avg_records:.2f}")

# -----------------------------------------------------------------------------
# 3. MISSING VALUES REPORT
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 2: MISSING VALUES REPORT")
print("="*80)

missing_report = pd.DataFrame({
    'Missing Values': df[['NDVI', 'NDWI', 'EVI']].isnull().sum(),
    'Missing Percentage (%)': (df[['NDVI', 'NDWI', 'EVI']].isnull().sum() / len(df)) * 100
})
print(missing_report)

print("\nNOTE:")
print("- No missing values are expected after preprocessing.")
print("- This section confirms that interpolation was successfully applied.")

# -----------------------------------------------------------------------------
# 4. SCIENTIFIC RANGE VALIDATION
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 3: SCIENTIFIC RANGE VALIDATION")
print("="*80)

for col in ['NDVI', 'NDWI', 'EVI']:
    valid_mask = df[col].notna()
    out_of_range = ((df[col] < -1) | (df[col] > 1)) & valid_mask
    count_bad = out_of_range.sum()
    total_non_null = valid_mask.sum()
    pct_bad = (count_bad / total_non_null) * 100 if total_non_null > 0 else 0

    print(f"\n{col}:")
    print(f"  Non-missing values: {total_non_null:,}")
    print(f"  Out-of-range values [-1, 1]: {count_bad:,}")
    print(f"  Out-of-range percentage: {pct_bad:.4f}%")

print("\nNOTE:")
print("- All indices are expected to remain within the scientific range [-1, 1] after cleaning.")

# -----------------------------------------------------------------------------
# 5. DESCRIPTIVE STATISTICS
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 4: DESCRIPTIVE STATISTICS")
print("="*80)

stats = df[['NDVI', 'NDWI', 'EVI']].describe().T
print(stats[['min', '25%', '50%', 'mean', '75%', 'max', 'std']])

print("\nINTERPRETATION NOTE:")
print("- Mean and median should now be consistent after outlier removal.")

# -----------------------------------------------------------------------------
# 6. CORRELATION ANALYSIS
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 5: CORRELATION ANALYSIS")
print("="*80)

corr_matrix = df[['NDVI', 'NDWI', 'EVI']].corr()
print(corr_matrix)

plt.figure(figsize=(6, 5))
plt.imshow(corr_matrix, interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title("Correlation Matrix: NDVI / NDWI / EVI")
for i in range(len(corr_matrix.index)):
    for j in range(len(corr_matrix.columns)):
        plt.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha='center', va='center')
plt.tight_layout()
plt.show()

print("\nINTERPRETATION NOTE:")
print("- NDVI and EVI are expected to have strong positive correlation.")
print("- NDWI may also correlate positively but usually less strongly.")

# -----------------------------------------------------------------------------
# 7. MONTHLY SEASONALITY CHECK
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 6: MONTHLY SEASONALITY CHECK")
print("="*80)

df['month'] = df['date'].dt.month

monthly_avg = df.groupby('month')[['NDVI', 'NDWI', 'EVI']].mean()

print("Monthly Average Values:")
print(monthly_avg)

plt.figure(figsize=(12, 6))
plt.plot(monthly_avg.index, monthly_avg['NDVI'], marker='o', label='NDVI')
plt.plot(monthly_avg.index, monthly_avg['NDWI'], marker='s', label='NDWI')
plt.plot(monthly_avg.index, monthly_avg['EVI'], marker='^', label='EVI')
plt.xticks(range(1, 13))
plt.xlabel("Month")
plt.ylabel("Average Index Value")
plt.title("Monthly Seasonality Pattern of NDVI, NDWI, and EVI")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

print("\nINTERPRETATION NOTE:")
print("- A meaningful seasonal pattern should appear across months.")
print("- Vegetation indices should rise and fall according to agricultural cycles.")

# -----------------------------------------------------------------------------
# 8. SAMPLE GRID TEMPORAL CONSISTENCY CHECK
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 7: SAMPLE GRID TEMPORAL CONSISTENCY CHECK")
print("="*80)

sample_grids = df['Grid_ID'].drop_duplicates().sample(5, random_state=42).tolist()
print("Sample Grids Selected:")
print(sample_grids)

for gid in sample_grids:
    sub = df[df['Grid_ID'] == gid].sort_values('date')

    plt.figure(figsize=(14, 4))
    plt.plot(sub['date'], sub['NDVI'], marker='o', label='NDVI')
    plt.plot(sub['date'], sub['NDWI'], marker='s', label='NDWI')
    plt.plot(sub['date'], sub['EVI'], marker='^', label='EVI')
    plt.title(f"Cleaned Time Series for Grid: {gid}")
    plt.xlabel("Date")
    plt.ylabel("Index Value")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

print("\nINTERPRETATION NOTE:")
print("- Time series should now appear smooth and free from abnormal EVI spikes.")

# -----------------------------------------------------------------------------
# 9. FINAL EVI RANGE VALIDATION
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 8: FINAL EVI RANGE VALIDATION")
print("="*80)

evi_outliers = df[(df['EVI'] < -1) | (df['EVI'] > 1)]

print(f"Remaining EVI Outliers After Cleaning: {len(evi_outliers):,}")

if len(evi_outliers) > 0:
    print("\nSample of Remaining EVI Outliers:")
    print(evi_outliers[['Grid_ID', 'date', 'EVI']].head(10))
else:
    print("No EVI outliers detected outside [-1, 1].")

print("\nNOTE:")
print("- All EVI outliers should have been removed after preprocessing.")
print("- This section confirms the success of the cleaning pipeline.")

# -----------------------------------------------------------------------------
# 10. GRID COMPLETENESS CHECK
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 9: GRID COMPLETENESS CHECK")
print("="*80)

grid_counts = df['Grid_ID'].value_counts()

print("Records per Grid Summary:")
print(grid_counts.describe())

expected_steps = 120
incomplete_grids = grid_counts[grid_counts != expected_steps]

print(f"\nExpected Records per Grid: {expected_steps}")
print(f"Grids NOT matching expected count: {len(incomplete_grids)}")

if len(incomplete_grids) > 0:
    print("\nSample incomplete grids:")
    print(incomplete_grids.head(10))
else:
    print("All grids have the expected number of temporal records.")

# -----------------------------------------------------------------------------
# 11. FINAL VALIDATION SUMMARY
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("SECTION 10: FINAL VALIDATION SUMMARY")
print("="*80)

print("1. The dataset structure has been checked.")
print("2. Missing values were successfully handled.")
print("3. Scientific ranges were validated.")
print("4. Correlation between NDVI, NDWI, and EVI was examined.")
print("5. Monthly seasonality was checked.")
print("6. Sample time-series consistency across multiple grids was visualized.")
print("7. EVI outliers were successfully removed and validated.")
print("8. Grid temporal completeness was verified.")

print("\nFINAL NOTE:")
print("This is the final post-cleaning validation audit.")
print("The dataset is now ready for feature engineering, climate merging, and machine learning.")

#**Final Approved Satellite Time-Series Dataset**

File Name:
Kafr_ElSheikh_True1km_TimeSeries_2020_2024_Cleaned.csv

This is the final cleaned and validated satellite dataset.
It will be used for:
1. Climate data merging
2. Feature engineering
3. Machine learning

#**Climate**

## **Climate Data Preprocessing**

The climate dataset was cleaned by handling missing dates and values using interpolation and forward/backward filling.  
Basic validation and visualization were performed to ensure data quality before merging with satellite data.

In [ ]:
# =============================================================================
# 1. Load Data
# =============================================================================
climate = pd.read_csv('climate_data.csv')

print("Data loaded successfully!")
print("Shape:", climate.shape)

# =============================================================================
# 2. Check columns (important!)
# =============================================================================
print("\nColumns:")
print(climate.columns)

# =============================================================================
# 3. Date Preparation
# =============================================================================
climate['date'] = pd.to_datetime(climate['date'])
climate = climate.sort_values('date').reset_index(drop=True)

# =============================================================================
# 4. Initial Checks
# =============================================================================
print("\n--- INITIAL CHECKS ---")

print("\nMissing Values:")
print(climate.isnull().sum())

print("\nDuplicates:", climate.duplicated().sum())

print("\nDate Range:")
print(climate['date'].min(), "→", climate['date'].max())

# =============================================================================
# 5. Check Missing Dates
# =============================================================================
full_range = pd.date_range(
    start=climate['date'].min(),
    end=climate['date'].max(),
    freq='D'
)

missing_dates = full_range.difference(climate['date'])
print("\nMissing Dates Count:", len(missing_dates))

# =============================================================================
# 6. Fill Missing Dates (reindex)
# =============================================================================
climate = (
    climate.set_index('date')
    .reindex(full_range)
    .rename_axis('date')
    .reset_index()
)

print("\nAfter reindexing (missing dates handled)")

# =============================================================================
# 7. Interpolation (Fill Missing Values)
# =============================================================================
cols = ['temperature','rainfall','humidity','wind_speed']

# make sure columns are numeric
climate[cols] = climate[cols].apply(pd.to_numeric, errors='coerce')

# interpolate
climate[cols] = climate[cols].interpolate(method='linear')

# fill edge cases
climate[cols] = climate[cols].ffill().bfill()

print("\nMissing values after filling:")
print(climate.isnull().sum())

# =============================================================================
# 8. Descriptive Statistics
# =============================================================================
print("\n--- STATISTICS ---")
print(climate[cols].describe())

# =============================================================================
# 9. Visualization
# =============================================================================
plt.figure(figsize=(12,4))
plt.plot(climate['date'], climate['temperature'])
plt.title("Temperature Over Time")
plt.grid(True)
plt.show()

sns.boxplot(data=climate[cols])
plt.title("Boxplot of Climate Variables")
plt.show()

# =============================================================================
# 10. Final Validation
# =============================================================================
print("\n--- FINAL VALIDATION ---")

print("Missing Values:")
print(climate.isnull().sum())

print("\nDuplicates:", climate.duplicated().sum())

print("\nFinal Shape:", climate.shape)

## **Temporal Alignment between Satellite & Climate data**

**Data Alignment**

Before merging the satellite and climate datasets, date alignment was performed to ensure both datasets share the same temporal coverage.

The date columns were first converted to datetime format, then unique dates were compared between the two datasets.  
The climate dataset was filtered to include only the dates موجودة in the satellite dataset.

Finally, a consistency check was performed to confirm that both datasets contain identical date values, ensuring accurate and reliable merging.

In [ ]:
satellite = pd.read_csv('Kafr_ElSheikh_True1km_TimeSeries_2020_2024_Cleaned.csv')
satellite_date = satellite['date'].unique()
print(sorted(satellite_date))

In [ ]:
climate = pd.read_csv('climate_data.csv')
climate_dates = climate['date'].unique()
print(sorted(climate_dates))

In [ ]:
satellite['date'] = pd.to_datetime(satellite['date'])
climate['date'] = pd.to_datetime(climate['date'])

In [ ]:
climate_df = climate[climate['date'].isin(satellite['date'])]

In [ ]:
sat_dates = set(satellite['date'])
clim_dates = set(climate_df['date'])
print(sat_dates == clim_dates)

# **Dataset Merging (Satellite + Climate)**

**Data Merging**

The satellite and climate datasets were merged on the date column after alignment.  
The final dataset was validated and saved for further processing.

In [ ]:
# convert dates (important)
satellite['date'] = pd.to_datetime(satellite['date'])
climate['date'] = pd.to_datetime(climate['date'])

climate_df = climate[climate['date'].isin(satellite['date'])]

merged_df = pd.merge(
    satellite,
    climate_df,
    on='date',
    how='inner'
)

# check
print(merged_df.head())
print(merged_df.shape)
print(merged_df.isna().sum())
merged_df.to_csv("Satellite_Climate_Merged_Clean.csv", index=False)

#**Post-Merge Data Validation & Exploratory Analysis**



A comprehensive analysis was performed on the merged dataset to understand its structure, quality, and relationships.

This included checking dataset size, missing values, and validating index ranges.  
Statistical summaries and correlation analysis were used to explore relationships between satellite and climate features.  
Additionally, seasonal patterns and sample grid trends were visualized to confirm temporal consistency.

In [ ]:
# =========================
# DATA OVERVIEW
# =========================

print("Total Rows:", len(merged_df))
print("Unique Grids:", merged_df['Grid_ID'].nunique())
print("Unique Dates:", merged_df['date'].nunique())

print("Date Range:")
print(merged_df['date'].min(), "→", merged_df['date'].max())

# =========================
# MISSING VALUES
# =========================

print("Missing Values:")
print(merged_df.isnull().sum())

# =========================
# RANGE VALIDATION
# =========================

for col in ['NDVI','NDWI','EVI']:
    out_of_range = merged_df[(merged_df[col] < -1) | (merged_df[col] > 1)]

    print(f"{col} out of range count:", len(out_of_range))

# =========================
# STATISTICS
# =========================

print(merged_df[['NDVI','NDWI','EVI','temperature','rainfall','humidity','wind_speed']].describe())
# =========================
# CORRELATION MATRIX
# =========================

corr = merged_df.corr(numeric_only=True)

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix (Satellite + Climate)")
plt.show()

# =========================
# MONTHLY SEASONALITY
# =========================

merged_df['month'] = merged_df['date'].dt.month

monthly_avg = merged_df.groupby('month')[['NDVI','temperature','rainfall']].mean()

plt.figure(figsize=(10,5))
plt.plot(monthly_avg.index, monthly_avg['NDVI'], label='NDVI')
plt.plot(monthly_avg.index, monthly_avg['temperature'], label='Temperature')
plt.plot(monthly_avg.index, monthly_avg['rainfall'], label='Rainfall')

plt.legend()
plt.title("Monthly Patterns")
plt.show()
# =========================
# SAMPLE GRID
# =========================

sample = merged_df[merged_df['Grid_ID'] == merged_df['Grid_ID'].iloc[0]]

plt.figure(figsize=(12,4))
plt.plot(sample['date'], sample['NDVI'], label='NDVI')
plt.plot(sample['date'], sample['temperature'], label='Temperature')

plt.legend()
plt.title("NDVI vs Temperature Over Time")
plt.show()

# **Feature Engineering**

### **Before Feature Engineering**

In [ ]:
merged_df = pd.read_csv('Satellite_Climate_Merged_Clean.csv')

print("merged_df loaded successfully")
print(merged_df.shape)
merged_df.head()

In [ ]:
df = merged_df
df = df.sort_values(['Grid_ID', 'date'])

# =========================
# Time Features
# =========================
df['date'] = pd.to_datetime(df['date'])

df['month'] = df['date'].dt.month
df['day_of_year'] = df['date'].dt.dayofyear
df['quarter'] = df['date'].dt.quarter


In [ ]:
df.head(10)

# **Feature Engineer**

## **NDVI Feature Engineering**


In [ ]:
print(df['NDVI'].describe())

In [ ]:
# Rolling
df['NDVI_roll_mean_3'] = df.groupby('Grid_ID')['NDVI'] \
    .transform(lambda x: x.rolling(3, min_periods=1).mean())

df['NDVI_roll_std_3'] = df.groupby('Grid_ID')['NDVI'] \
    .transform(lambda x: x.rolling(3, min_periods=1).std()) \
    .fillna(0)

# Lag
df['NDVI_lag1'] = df.groupby('Grid_ID')['NDVI'].shift(1).fillna(0)

# Change
df['NDVI_change'] = df.groupby('Grid_ID')['NDVI'].diff().fillna(0)

# EVI + NDWI
df['EVI_change'] = df.groupby('Grid_ID')['EVI'].diff().fillna(0)
df['NDWI_change'] = df.groupby('Grid_ID')['NDWI'].diff().fillna(0)

# Climate features
df['temp_avg_3'] = df.groupby('Grid_ID')['temperature'] \
    .transform(lambda x: x.rolling(3, min_periods=1).mean())

df['rain_sum_3'] = df.groupby('Grid_ID')['rainfall'] \
    .transform(lambda x: x.rolling(3, min_periods=1).sum())

# NDVI class (optional)
df['NDVI_class'] = pd.cut(
    df['NDVI'],
    bins=[-1, 0.1, 0.3, 0.5, 0.7, 1],
    labels=['bare', 'sparse', 'moderate', 'good', 'excellent']
)
df['NDVI_class'] = df['NDVI_class'].cat.codes

# Anomaly (fixed)
df['NDVI_anomaly'] = (
    df['NDVI'] - df.groupby(['Grid_ID', 'month'])['NDVI'].transform('mean')
) / (
    df.groupby(['Grid_ID', 'month'])['NDVI'].transform('std') + 1e-6
)

## **NDVI Temporal Dynamics**

In [ ]:
# NDVI Change
df['NDVI_change'] = df.groupby('Grid_ID')['NDVI'].diff().fillna(0)

# NDVI Change Variability (FIXED)
df['NDVI_change_std'] = df.groupby('Grid_ID')['NDVI_change'] \
    .transform(lambda x: x.rolling(3, min_periods=1).std()) \
    .fillna(1e-6)

# NDVI Trend (numeric)
df['NDVI_trend'] = np.where(
    df['NDVI_change'] > df['NDVI_change_std'], 1,
    np.where(df['NDVI_change'] < -df['NDVI_change_std'], -1, 0)
)

# NDVI Anomaly Flag
df['NDVI_anomaly_flag'] = (
    df['NDVI_change'].abs() > 2 * df['NDVI_change_std']
).astype(int)

# Smart stress feature
df['stress_event'] = (
    (df['NDVI_change'] < -df['NDVI_change_std']) &
    (df['rain_sum_3'] < df['rain_sum_3'].median())
).astype(int)

## **NDWI Features**

In [ ]:
print(df['NDWI'].describe())

In [ ]:
# Rolling mean
df['NDWI_roll_mean_3'] = df.groupby('Grid_ID')['NDWI'] \
    .transform(lambda x: x.rolling(3, min_periods=1).mean())

# Lag
df['NDWI_lag1'] = df.groupby('Grid_ID')['NDWI'].shift(1).fillna(0)

# Change
df['NDWI_change'] = df.groupby('Grid_ID')['NDWI'].diff().fillna(0)

# Interaction
df['NDVI_NDWI_diff'] = df['NDVI'] - df['NDWI']
df['NDVI_NDWI_ratio'] = df['NDVI'] / (df['NDWI'] + 1e-6)

# Seasonal stats (FIXED)
df['NDWI_month_mean'] = df.groupby(['Grid_ID', 'month'])['NDWI'].transform('mean')

df['NDWI_month_std'] = df.groupby(['Grid_ID', 'month'])['NDWI'] \
    .transform('std').fillna(1e-6)

# Water stress (improved)
df['water_stress'] = (
    (df['NDWI'] < (df['NDWI_month_mean'] - 1.5 * df['NDWI_month_std'])) &
    (df['NDVI'] < df['NDVI_roll_mean_3'])
).astype(int)

## **EVI Features**

In [ ]:
print(df['EVI'].describe())

In [ ]:
# Rolling mean
df['EVI_roll_mean_3'] = df.groupby('Grid_ID')['EVI'] \
    .transform(lambda x: x.rolling(3, min_periods=1).mean())

# Lag
df['EVI_lag1'] = df.groupby('Grid_ID')['EVI'].shift(1).fillna(0)

# Change
df['EVI_change'] = df.groupby('Grid_ID')['EVI'].diff().fillna(0)

# Stability
df['EVI_stability'] = df.groupby('Grid_ID')['EVI'] \
    .transform(lambda x: x.rolling(3, min_periods=1).std()).fillna(0)

# Interaction
df['NDVI_EVI_diff'] = df['NDVI'] - df['EVI']
df['NDVI_EVI_ratio'] = df['NDVI'] / (df['EVI'] + 1e-6)

# Improved vegetation health
df['veg_health'] = (
    df['NDVI_roll_mean_3'] + df['EVI_roll_mean_3']
) / 2

# EVI anomaly
df['EVI_anomaly'] = (
    df['EVI'] - df.groupby(['Grid_ID', 'month'])['EVI'].transform('mean')
) / (
    df.groupby(['Grid_ID', 'month'])['EVI'].transform('std') + 1e-6
)

## **Temperature Features**

In [ ]:
print(df['temperature'].describe())

In [ ]:
# Lag
df['temp_lag1'] = df.groupby('Grid_ID')['temperature'].shift(1).fillna(0)

# Change
df['temp_change'] = df.groupby('Grid_ID')['temperature'].diff().fillna(0)

# Rolling
df['temp_avg_3'] = df.groupby('Grid_ID')['temperature'] \
    .transform(lambda x: x.rolling(3, min_periods=1).mean())

# Seasonal stats (FIXED)
df['temp_month_mean'] = df.groupby(['Grid_ID', 'month'])['temperature'].transform('mean')

df['temp_month_std'] = df.groupby(['Grid_ID', 'month'])['temperature'] \
    .transform('std').fillna(1e-6)

# Stress features (improved)
df['heat_stress'] = (
    (df['temperature'] > (df['temp_month_mean'] + 1.5 * df['temp_month_std'])) &
    (df['NDVI'] < df['NDVI_roll_mean_3'])
).astype(int)

df['cold_stress'] = (
    (df['temperature'] < (df['temp_month_mean'] - 1.5 * df['temp_month_std'])) &
    (df['NDVI'] < df['NDVI_roll_mean_3'])
).astype(int)

# Combined stress
df['climate_stress'] = (
    (df['heat_stress'] == 1) |
    (df['cold_stress'] == 1)
).astype(int)

# Interaction
df['temp_NDVI'] = df['temperature'] * df['NDVI']
df['temp_NDVI_ratio'] = df['temperature'] / (df['NDVI'] + 1e-6)

In [ ]:
# Rolling mean (FIXED - no leakage)
df['temp_avg_3'] = df.groupby('Grid_ID')['temperature'] \
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())

# Rolling std (FIXED)
df['temp_roll_std_3'] = df.groupby('Grid_ID')['temperature'] \
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).std()) \
    .fillna(1e-6)

# Anomaly
df['temp_anomaly'] = df['temperature'] - df['temp_avg_3']

# Normalized anomaly
df['temp_anomaly_norm'] = df['temp_anomaly'] / df['temp_roll_std_3']

# Extreme temperature
df['temp_extreme'] = (df['temp_anomaly_norm'].abs() > 2).astype(int)

# Smart stress event
df['temp_stress_event'] = (
    (df['temp_anomaly_norm'] > 2) &
    (df['NDVI_change'] < 0)
).astype(int)

## **Rainfall Features**

In [ ]:
print(df['rainfall'].describe())

In [ ]:
# Lag
df['rain_lag1'] = df.groupby('Grid_ID')['rainfall'].shift(1).fillna(0)

# Change
df['rain_change'] = df.groupby('Grid_ID')['rainfall'].diff().fillna(0)

# Binary
df['has_rain'] = (df['rainfall'] > 0).astype(int)

# Rolling rainfall (FIXED)
df['rain_sum_3'] = df.groupby('Grid_ID')['rainfall'] \
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).sum())

# Rain streak
df['rain_streak'] = df.groupby('Grid_ID')['has_rain'] \
    .transform(lambda x: x.rolling(3, min_periods=1).sum())

# Interaction
df['rain_NDVI'] = df['rainfall'] * df['NDVI']
df['rain_NDVI_ratio'] = df['rainfall'] / (df['NDVI'] + 1e-6)

# Drought (adaptive)
df['drought'] = (df['rain_sum_3'] < df['rain_sum_3'].median()).astype(int)

# Drought severity
df['drought_severity'] = np.log1p(1 / (df['rain_sum_3'] + 1e-3))

# Rainfall anomaly (FIXED)
df['rainfall_anomaly'] = (
    df['rainfall'] - df.groupby(['Grid_ID', 'month'])['rainfall'].transform('mean')
) / (
    df.groupby(['Grid_ID', 'month'])['rainfall'].transform('std') + 1e-6
)

## **Humidity & VPD Features**

In [ ]:
print(df['humidity'].describe())

In [ ]:
# Rolling humidity (FIXED)
df['humidity_avg_3'] = df.groupby('Grid_ID')['humidity'] \
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())

# Lag
df['humidity_lag1'] = df.groupby('Grid_ID')['humidity'].shift(1).fillna(0)

# Clip
df['humidity'] = df['humidity'].clip(0, 100)

# Change
df['humidity_change'] = df.groupby('Grid_ID')['humidity'].diff().fillna(0)

# VPD
df['VPD'] = (1 - df['humidity'] / 100) * 0.611 * np.exp(
    17.27 * df['temperature'] / (df['temperature'] + 237.3)
)

# Seasonal normalization (FIXED)
df['VPD_month_mean'] = df.groupby(['Grid_ID', 'month'])['VPD'].transform('mean')

df['VPD_month_std'] = df.groupby(['Grid_ID', 'month'])['VPD'] \
    .transform('std').fillna(1e-6)

df['VPD_norm'] = (df['VPD'] - df['VPD_month_mean']) / df['VPD_month_std']

# Stress features
df['VPD_stress'] = (df['VPD_norm'] > 1.5).astype(int)

df['dry_stress'] = (
    (df['VPD_norm'] > 1.5) &
    (df['NDVI_change'] < 0)
).astype(int)

# Interaction
df['humidity_temp_ratio'] = df['humidity'] / (df['temperature'] + 1e-6)

## **Wind & Evapotranspiration Features**


In [ ]:
print(df['wind_speed'].describe())

In [ ]:
# Rolling wind (FIXED)
df['wind_avg_3'] = df.groupby('Grid_ID')['wind_speed'] \
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())

# Lag
df['wind_lag1'] = df.groupby('Grid_ID')['wind_speed'].shift(1).fillna(0)

# Change
df['wind_change'] = df.groupby('Grid_ID')['wind_speed'].diff().fillna(0)

# High wind (dynamic)
df['wind_q75'] = df.groupby('Grid_ID')['wind_speed'] \
    .transform(lambda x: x.shift(1).rolling(10, min_periods=3).quantile(0.75))

df['high_wind'] = (df['wind_speed'] > df['wind_q75']).astype(int)

# ET
df['ET'] = df['temperature'] * (1 + df['wind_speed'] / 10) * (1 - df['humidity'] / 100)

# Seasonal normalization (FIXED)
df['ET_mean'] = df.groupby(['Grid_ID', 'month'])['ET'].transform('mean')

df['ET_std'] = df.groupby(['Grid_ID', 'month'])['ET'] \
    .transform('std').fillna(1e-6)

df['ET_norm'] = (df['ET'] - df['ET_mean']) / df['ET_std']

# Stress features
df['ET_stress'] = (df['ET_norm'] > 1.5).astype(int)

df['evap_stress'] = (
    (df['ET_norm'] > 1.5) &
    (df['NDVI_change'] < 0)
).astype(int)

# Interaction
df['wind_temp'] = df['wind_speed'] * df['temperature']

# **Add season feature**

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df['season'] = df['month'].apply(get_season)

print("Season feature added successfully.")
print(df[['month', 'season']].head(10))

#### **Target**

Additional time-based features were created from the date to capture seasonal patterns.  
Changes in vegetation indices (EVI, NDWI) were also calculated to reflect short-term variations.  
Finally, a binary target variable (stress_label) was defined based on NDVI threshold.

In [ ]:
# Time Features
df['day_of_year'] = df['date'].dt.dayofyear
df['day_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

df['month'] = df['date'].dt.month

# Vegetation change
df['EVI_change'] = df.groupby('Grid_ID')['EVI'].diff().fillna(0)
df['NDWI_change'] = df.groupby('Grid_ID')['NDWI'].diff().fillna(0)

In [ ]:
df.isna().sum()

In [ ]:
print(df.shape)

In [ ]:
df.to_csv('Feature_Engineered_Dataset.csv', index=False)

# **Validate Feature Engineering**

In [ ]:
print(df.columns.tolist())
print("Total Columns:", len(df.columns))

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]

print("===== Missing Features =====")
print(missing.sort_values(ascending=False))

In [ ]:
print(df.describe())

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns

outlier_summary = pd.DataFrame({
    'min': df[numeric_cols].min(),
    'max': df[numeric_cols].max(),
    'mean': df[numeric_cols].mean(),
    'std': df[numeric_cols].std()
})

print(outlier_summary)

In [ ]:
print(df[['drought', 'season']].head())
print(df['drought'].value_counts())

# **Handle missing values**

In [ ]:
# Rolling / average features
df['temp_avg_3'] = df['temp_avg_3'].fillna(df['temperature'])
df['rain_sum_3'] = df['rain_sum_3'].fillna(df['rainfall'])
df['humidity_avg_3'] = df['humidity_avg_3'].fillna(df['humidity'])
df['wind_avg_3'] = df['wind_avg_3'].fillna(df['wind_speed'])

# Anomaly features
df['temp_anomaly'] = df['temp_anomaly'].fillna(0)
df['temp_anomaly_norm'] = df['temp_anomaly_norm'].fillna(0)

# Severity / quantile features
df['drought_severity'] = df['drought_severity'].fillna(0)
df['wind_q75'] = df['wind_q75'].fillna(df['wind_speed'])

print("Missing values handled successfully.")
print(df.isnull().sum()[df.isnull().sum() > 0])

# **Final check after preprocessing**

In [ ]:
# =========================
# Check basic dataset information
# =========================

print("Dataset shape:")
print(df.shape)

print("\nColumn types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# =========================
# Check missing values after preprocessing
# =========================

total_missing = df.isnull().sum().sum()

print("Total missing values:")
print(total_missing)

print("\nMissing values by column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# =========================
# Validate anomaly features
# =========================

anomaly_cols = [
    'NDVI_anomaly',
    'rainfall_anomaly',
    'temp_anomaly'
]

print(df[anomaly_cols].describe())

print("\nMeans:")
for col in anomaly_cols:
    print(f"{col}: {df[col].mean():.6f}")

In [ ]:
# =========================
# Validate normalized and climate features
# =========================

check_cols = [
    'temp_anomaly_norm',
    'VPD',
    'VPD_norm',
    'ET',
    'ET_norm'
]

print(df[check_cols].describe())

In [ ]:
# =========================
# Validate binary and categorical features
# =========================

print("Drought distribution:")
print(df['drought'].value_counts())

print("\nSeason distribution:")
print(df['season'].value_counts())

print("\nHigh wind distribution:")
print(df['high_wind'].value_counts())

In [ ]:
# =========================
# Visual distribution check
# =========================

import matplotlib.pyplot as plt

cols_to_plot = [
    'NDVI_anomaly',
    'rainfall_anomaly',
    'temp_anomaly',
    'VPD',
    'ET'
]

for col in cols_to_plot:
    plt.figure(figsize=(6,4))
    plt.hist(df[col], bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [ ]:
# =========================
# Final ready check before risk score
# =========================

print("Final dataset shape:", df.shape)
print("Total missing values:", df.isnull().sum().sum())

print("\nKey features preview:")
display(df[[
    'NDVI_anomaly',
    'rainfall_anomaly',
    'temp_anomaly',
    'VPD',
    'ET',
    'drought',
    'season'
]].head())

# **Fix temp_anomaly_norm**

In [ ]:
# =========================
# Fix temp_anomaly_norm
# =========================

df['temp_anomaly_norm'] = (
    df['temp_anomaly'] - df['temp_anomaly'].mean()
) / (df['temp_anomaly'].std() + 1e-6)

print("temp_anomaly_norm fixed successfully.")
print(df['temp_anomaly_norm'].describe())

In [ ]:
# =========================
# Validate temperature features
# =========================

print("temp_anomaly statistics:")
print(df['temp_anomaly'].describe())

print("\ntemp_anomaly mean:")
print(df['temp_anomaly'].mean())

print("\ntemp_anomaly_norm statistics:")
print(df['temp_anomaly_norm'].describe())

print("\ntemp_anomaly_norm mean:")
print(df['temp_anomaly_norm'].mean())

print("\ntemp_anomaly_norm std:")
print(df['temp_anomaly_norm'].std())

In [ ]:
# =========================
# Plot fixed temp_anomaly_norm
# =========================

import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
plt.hist(df['temp_anomaly_norm'], bins=30)
plt.title("Distribution of temp_anomaly_norm")
plt.xlabel("temp_anomaly_norm")
plt.ylabel("Count")
plt.show()

In [ ]:
# =========================
# Final sanity check for key features
# =========================

key_cols = [
    'NDVI_anomaly',
    'rainfall_anomaly',
    'temp_anomaly',
    'temp_anomaly_norm',
    'VPD',
    'ET',
    'drought',
    'season'
]

print(df[key_cols].describe(include='all'))
print("\nTotal missing values in key features:")
print(df[key_cols].isnull().sum())

# **Add seasonal multipliers**

In [ ]:
temp_multiplier_map = {
    'Winter': 1.0,
    'Spring': 1.1,
    'Summer': 1.3,
    'Autumn': 1.0
}

drought_multiplier_map = {
    'Winter': 0.9,
    'Spring': 1.1,
    'Summer': 1.3,
    'Autumn': 1.0
}

df['temp_multiplier'] = df['season'].map(temp_multiplier_map)
df['drought_multiplier'] = df['season'].map(drought_multiplier_map)

print("Seasonal multipliers added successfully.")
display(df[['season', 'temp_multiplier', 'drought_multiplier']].head(10))

# **Build raw risk components**

In [ ]:
# 1) Vegetation risk
# Negative NDVI anomaly means vegetation is worse than seasonal normal
neg_ndvi = (-df['NDVI_anomaly']).clip(lower=0)
df['veg_risk'] = neg_ndvi / (neg_ndvi.max() + 1e-6)

# 2) Rainfall risk
# Negative rainfall anomaly means rainfall is below normal
neg_rain = (-df['rainfall_anomaly']).clip(lower=0)
df['rain_risk'] = neg_rain / (neg_rain.max() + 1e-6)

# 3) Temperature risk
# Positive temperature anomaly means hotter than normal
pos_temp = df['temp_anomaly'].clip(lower=0)
df['temp_risk'] = pos_temp / (pos_temp.max() + 1e-6)

# 4) VPD risk
# Higher VPD means stronger atmospheric water demand
df['vpd_risk'] = df['VPD'] / (df['VPD'].max() + 1e-6)

# 5) Drought risk
# Binary drought feature
df['drought_risk'] = df['drought'].astype(float)

print("Raw risk components created successfully.")
display(df[['veg_risk', 'rain_risk', 'temp_risk', 'vpd_risk', 'drought_risk']].head(10))

# **Apply seasonal adjustment**

In [ ]:
df['temp_risk_adjusted'] = (df['temp_risk'] * df['temp_multiplier']).clip(upper=1)
df['drought_risk_adjusted'] = (df['drought_risk'] * df['drought_multiplier']).clip(upper=1)

print("Season-adjusted components created successfully.")
display(df[[
    'season',
    'temp_risk',
    'temp_multiplier',
    'temp_risk_adjusted',
    'drought_risk',
    'drought_multiplier',
    'drought_risk_adjusted'
]].head(10))

# **Validate all risk components**

In [ ]:
risk_component_cols = [
    'veg_risk',
    'rain_risk',
    'temp_risk',
    'temp_risk_adjusted',
    'vpd_risk',
    'drought_risk',
    'drought_risk_adjusted'
]

print(df[risk_component_cols].describe())

print("\nMinimum values:")
print(df[risk_component_cols].min())

print("\nMaximum values:")
print(df[risk_component_cols].max())

# **Create final season-aware risk score**

In [ ]:
df['risk_score'] = (
    0.35 * df['veg_risk'] +
    0.20 * df['rain_risk'] +
    0.15 * df['temp_risk_adjusted'] +
    0.15 * df['vpd_risk'] +
    0.15 * df['drought_risk_adjusted']
) * 100

print("Risk score created successfully.")
print(df['risk_score'].describe())

# **Create risk labels**

In [ ]:
def classify_risk(score):
    if score < 20:
        return 'Low'
    elif score < 40:
        return 'Medium'
    else:
        return 'High'

df['risk_label'] = df['risk_score'].apply(classify_risk)

print(df['risk_label'].value_counts())

# **Visualize risk score distribution**

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['risk_score'], bins=30)
plt.xlabel("Risk Score")
plt.ylabel("Count")
plt.title("Distribution of Risk Score")
plt.show()

# **Visualize risk label distribution**

In [ ]:
df['risk_label'].value_counts().plot(kind='bar', figsize=(7, 4))
plt.title("Distribution of Risk Labels")
plt.xlabel("Risk Label")
plt.ylabel("Count")
plt.show()

# **Preview final risk output**

In [ ]:
display(df[[
    'Grid_ID',
    'date',
    'season',
    'NDVI_anomaly',
    'rainfall_anomaly',
    'temp_anomaly',
    'VPD',
    'drought',
    'veg_risk',
    'rain_risk',
    'temp_risk_adjusted',
    'vpd_risk',
    'drought_risk_adjusted',
    'risk_score',
    'risk_label'
]].head(10))

# **Average risk score by season**

In [ ]:
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']

avg_risk_by_season = (
    df.groupby('season')['risk_score']
      .mean()
      .reindex(season_order)
)

avg_risk_by_season.plot(kind='bar', figsize=(7,4))
plt.title("Average Risk Score by Season")
plt.xlabel("Season")
plt.ylabel("Average Risk Score")
plt.show()

# **Average risk score by month**

In [ ]:
avg_risk_by_month = df.groupby('month')['risk_score'].mean()

avg_risk_by_month.plot(figsize=(8,4))
plt.title("Average Risk Score by Month")
plt.xlabel("Month")
plt.ylabel("Average Risk Score")
plt.show()

# **Average score by risk label**

In [ ]:
label_order = ['Low', 'Medium', 'High']

avg_score_by_label = (
    df.groupby('risk_label')['risk_score']
      .mean()
      .reindex(label_order)
)

avg_score_by_label.plot(kind='bar', figsize=(7,4))
plt.title("Average Risk Score by Risk Label")
plt.xlabel("Risk Label")
plt.ylabel("Average Risk Score")
plt.show()

# **Top 10 highest-risk grids**

In [ ]:
top_grids = (
    df.groupby('Grid_ID')['risk_score']
      .mean()
      .sort_values(ascending=False)
      .head(10)
)

top_grids.plot(kind='bar', figsize=(10,4))
plt.title("Top 10 Highest-Risk Grids")
plt.xlabel("Grid ID")
plt.ylabel("Average Risk Score")
plt.show()

# **Correlation of risk score with key factors**

In [ ]:
corr_cols = ['risk_score', 'NDVI_anomaly', 'rainfall_anomaly', 'temp_anomaly', 'VPD', 'drought']
print(df[corr_cols].corr())

In [ ]:
# =========================
# Correlation heatmap for risk score and key factors
# =========================

import matplotlib.pyplot as plt
import seaborn as sns

corr_cols = [
    'risk_score',
    'NDVI_anomaly',
    'rainfall_anomaly',
    'temp_anomaly',
    'VPD',
    'drought'
]

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Heatmap: Risk Score vs Key Factors")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Correlation of key factors with risk score
# =========================

corr_with_score = df[[
    'risk_score',
    'NDVI_anomaly',
    'rainfall_anomaly',
    'temp_anomaly',
    'VPD',
    'drought'
]].corr()['risk_score'].drop('risk_score')

plt.figure(figsize=(8, 5))
corr_with_score.sort_values().plot(kind='barh')
plt.title("Correlation of Key Factors with Risk Score")
plt.xlabel("Correlation")
plt.ylabel("Features")
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# **Risk Score Over Time**

In [ ]:
# =========================
# Select one grid for time-series analysis
# =========================

sample_grid_id = "G_294_3432"

grid_data = df[df['Grid_ID'] == sample_grid_id].sort_values('date').copy()

print("Selected Grid:", sample_grid_id)
print("Shape:", grid_data.shape)

display(grid_data.head())

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(grid_data['date'], grid_data['risk_score'], marker='o', linestyle='-', markersize=4, label='Risk Score')
plt.title(f'Risk Score Over Time - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Risk Score', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Smoothed risk score trend
# =========================

grid_data['risk_score_smooth'] = (
    grid_data['risk_score']
    .rolling(3, min_periods=1)
    .mean()
)

plt.figure(figsize=(15, 5))

plt.plot(
    grid_data['date'],
    grid_data['risk_score'],
    alpha=0.4,
    marker='o',
    markersize=3,
    label='Raw Risk Score'
)

plt.plot(
    grid_data['date'],
    grid_data['risk_score_smooth'],
    linewidth=2.5,
    label='Smoothed Trend'
)

plt.title(f'Risk Score Trend Over Time - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Risk Score', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()

# **Climate Anomalies Over Time**

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(grid_data['date'], grid_data['temp_anomaly'], marker='o', linestyle='-', markersize=4, label='Temp Anomaly')
plt.plot(grid_data['date'], grid_data['rainfall_anomaly'], marker='s', linestyle='-', markersize=4, label='Rainfall Anomaly')
plt.title(f'Climate Anomalies Over Time - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Anomaly Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# **VPD and ET Over Time**

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(grid_data['date'], grid_data['VPD'], marker='o', linestyle='-', markersize=4, label='VPD')
plt.plot(grid_data['date'], grid_data['ET'], marker='s', linestyle='-', markersize=4, label='ET')
plt.title(f'VPD and ET Over Time - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# **Risk Score Colored by Label**

In [ ]:
color_map = {'Low': 'green', 'Medium': 'orange', 'High': 'red'}

plt.figure(figsize=(15, 5))
for label in ['Low', 'Medium', 'High']:
    subset = grid_data[grid_data['risk_label'] == label]
    plt.scatter(subset['date'], subset['risk_score'],
                label=label, s=35, alpha=0.8,
                color=color_map[label])

plt.title(f'Risk Score by Label - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Risk Score', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# **High Risk Points Over Time**

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(grid_data['date'], grid_data['risk_score'], linestyle='-', alpha=0.7, label='Risk Score')

high_points = grid_data[grid_data['risk_label'] == 'High']
plt.scatter(high_points['date'], high_points['risk_score'],
            color='red', s=50, label='High Risk')

plt.title(f'High Risk Events Over Time - Grid: {sample_grid_id}', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Risk Score', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# **Create model dataset copy**

In [ ]:
df_model = df.copy()

print("Original dataset shape:", df.shape)
print("Model dataset shape:", df_model.shape)
print("Unique grids:", df_model['Grid_ID'].nunique())

In [ ]:
# =========================
# Verify time sequence before shift
# =========================

df_model = df_model.sort_values(['Grid_ID', 'date']).reset_index(drop=True)

print("Time sequence sorted successfully.")
display(df_model[['Grid_ID', 'date']].head(10))

# **Create future NDVI**

In [ ]:
df_model['NDVI_future'] = (
    df_model.groupby('Grid_ID')['NDVI']
    .shift(-1)
)

display(
    df_model[['Grid_ID', 'date', 'NDVI', 'NDVI_future']].head(10)
)

In [ ]:
# =========================
# Check rows before deletion
# =========================

rows_before = df_model.shape[0]

print("Rows before deletion:", rows_before)


In [ ]:
# =========================
# Check missing values before deletion
# =========================

missing_future = df_model['NDVI_future'].isna().sum()

unique_grids = df_model['Grid_ID'].nunique()

print("Missing values in NDVI_future:", missing_future)
print("Unique grids:", unique_grids)

if missing_future == unique_grids:
    print("Perfect: one missing future value per grid.")
else:
    print("Warning: mismatch detected.")

In [ ]:
# =========================
# Verify missing rows are last rows only
# =========================

missing_rows = df_model[df_model['NDVI_future'].isna()]

display(
    missing_rows[['Grid_ID', 'date', 'NDVI', 'NDVI_future']]
    .head(10)
)

print("Number of missing rows:", missing_rows.shape[0])

In [ ]:
# =========================
# Confirm missing rows are last dates
# =========================

last_dates = (
    df_model.groupby('Grid_ID')['date']
    .max()
    .reset_index()
)

check_last = missing_rows.merge(
    last_dates,
    on=['Grid_ID', 'date'],
    how='inner'
)

print("Missing rows count:", missing_rows.shape[0])
print("Matched last-date rows:", check_last.shape[0])

if missing_rows.shape[0] == check_last.shape[0]:
    print("Perfect: all missing rows are last rows only.")
else:
    print("Warning: some missing rows are not last rows.")

# **Delete last row from each grid**

In [ ]:
df_model = df_model.dropna(subset=['NDVI_future']).reset_index(drop=True)

rows_after = df_model.shape[0]

print("Rows after deletion:", rows_after)
print("Deleted rows:", rows_before - rows_after)

In [ ]:
# =========================
# Verify exactly one row deleted per grid
# =========================

expected_deleted = unique_grids
actual_deleted = rows_before - rows_after

print("Expected deleted rows:", expected_deleted)
print("Actual deleted rows:", actual_deleted)

if expected_deleted == actual_deleted:
    print("Perfect: exactly one row deleted from each grid.")
else:
    print("Warning: mismatch detected.")

# **Create future stress label**

In [ ]:
stress_threshold = 0.3

df_model['stress_label'] = (
    df_model['NDVI_future'] < stress_threshold
).astype(int)

print("Stress label created successfully.")
print(df_model['stress_label'].value_counts())

# **Validate risk score vs future stress**

In [ ]:
print(
    df_model.groupby('stress_label')['risk_score']
    .describe()
)

# **Mean risk score by future label**

In [ ]:
mean_scores = (
    df_model.groupby('stress_label')['risk_score']
    .mean()
)

print(mean_scores)

mean_scores.plot(kind='bar', figsize=(6,4))
plt.title("Average Risk Score by Future Stress Label")
plt.xlabel("Future Stress Label")
plt.ylabel("Average Risk Score")
plt.xticks([0,1], ['Healthy (0)', 'Stressed (1)'], rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

# **Check time interval consistency**

In [ ]:
df_model['days_diff'] = (
    df_model.groupby('Grid_ID')['date']
    .diff()
    .dt.days
)

print(df_model['days_diff'].describe())

print("\nUnique day intervals:")
print(df_model['days_diff'].value_counts().head(10))

# **Future stress rate by risk level validation**

In [ ]:
stress_rate_by_label = (
    df_model.groupby('risk_label')['stress_label']
    .mean()
    .reindex(['Low', 'Medium', 'High'])
) * 100

print(stress_rate_by_label)

stress_rate_by_label.plot(kind='bar', figsize=(7,4))
plt.title("Future Stress Rate by Risk Label")
plt.xlabel("Risk Label")
plt.ylabel("Stress Rate (%)")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

## **Validate risk score against future stress**

This validation step confirms that the engineered **risk score is strongly aligned with future crop stress patterns**.

The percentage of future stressed samples increases consistently across the risk levels:

- **Low Risk → 6.95%**
- **Medium Risk → 13.01%**
- **High Risk → 19.88%**

This clear increasing trend indicates that higher current risk scores are associated with a greater probability of crop stress in the next 15 days.

The results validate that the risk score is meaningful and can be reliably used as a predictive feature for the machine learning model.

In particular, the **High Risk level shows nearly 3× the future stress rate compared to the Low Risk level**, which confirms that the risk engineering pipeline is logically and statistically consistent.

# **Risk score distribution by future stress label**

In [ ]:
plt.figure(figsize=(7,5))
df_model.boxplot(column='risk_score', by='stress_label')

plt.title("Risk Score by Future Stress Label")
plt.suptitle("")
plt.xlabel("Future Stress Label")
plt.ylabel("Risk Score")
plt.xticks([1, 2], ['Healthy (0)', 'Stressed (1)'])

plt.show()

## **Validation of Risk Score Distribution**

The box plot confirms that the Risk Score is generally higher for future stressed cases (label = 1) compared to healthy cases (label = 0).

The median risk score for stressed samples is higher than healthy samples, which indicates that the engineered feature is aligned with the target label and can be used confidently in model training.

# **Save Agricultural Datasets**

In [ ]:
# ==========================================
# Save Directory (Works on Colab & Local)
# ==========================================

import os

if os.path.exists('/content/drive'):
    # Google Colab
    SAVE_DIR = "/content/drive/MyDrive/DEPI_Project"
else:
    # Local Jupyter / VS Code
    SAVE_DIR = "DEPI_Project"

os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================
# File Paths
# ==========================================

full_dataset_path = os.path.join(
    SAVE_DIR,
    "agricultural_full_dataset.csv"
)

training_dataset_path = os.path.join(
    SAVE_DIR,
    "agricultural_dataset_ready_for_training.csv"
)

# ==========================================
# Save Files
# ==========================================

df.to_csv(full_dataset_path, index=False)
df_model.to_csv(training_dataset_path, index=False)

print("Datasets saved successfully!")
print("Full Dataset:", full_dataset_path)
print("Training Dataset:", training_dataset_path)

# **Check Dataset Shapes**

In [ ]:
print("Agricultural Full Dataset Shape:")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\n" + "="*50)

print("Agricultural Dataset Ready for Training Shape:")
print(f"Rows: {df_model.shape[0]}")
print(f"Columns: {df_model.shape[1]}")

In [ ]:
# ==========================================
#  Print Column Names
# ==========================================

print("Columns in Agricultural Full Dataset:")
print(df.columns.tolist())

print("\n" + "="*80 + "\n")

print("Columns in Training Dataset:")
print(df_model.columns.tolist())

In [ ]:
# ==========================================
#  Compare Column Differences
# ==========================================

full_cols = set(df.columns)
training_cols = set(df_model.columns)

only_in_full = full_cols - training_cols
only_in_training = training_cols - full_cols

print("Columns only in Full Dataset:")
print(list(only_in_full))

print("\n" + "="*50 + "\n")

print("Columns only in Training Dataset:")
print(list(only_in_training))

In [ ]:
# file paths
SAVE_DIR = "/content/drive/MyDrive/DEPI PROJECT"

full_dataset_path = f"{SAVE_DIR}/agricultural_full_dataset.csv"
training_dataset_path = f"{SAVE_DIR}/agricultural_dataset_ready_for_training.csv"

# load datasets
df_full = pd.read_csv(full_dataset_path)
df_train = pd.read_csv(training_dataset_path)

# convert date column to datetime
df_full['date'] = pd.to_datetime(df_full['date'])
df_train['date'] = pd.to_datetime(df_train['date'])

print("Datasets loaded successfully")
print("Full dataset shape:", df_full.shape)
print("Training dataset shape:", df_train.shape)

In [ ]:
# =============================================================================
# DATASET OVERVIEW + MISSING VALUES CHECK
# =============================================================================

def dataset_overview(df, dataset_name):
    print("\n" + "="*80)
    print(dataset_name)
    print("="*80)

    print(f"Total Rows: {len(df):,}")
    print(f"Total Columns: {df.shape[1]}")
    print(f"Total Unique Grids: {df['Grid_ID'].nunique():,}")
    print(f"Total Unique Dates: {df['date'].nunique():,}")
    print(f"Start Date: {df['date'].min()}")
    print(f"End Date:   {df['date'].max()}")

    avg_records = df['Grid_ID'].value_counts().mean()
    print(f"Average Records per Grid: {avg_records:.2f}")

    print("\nMissing Values Per Column:")
    missing = df.isnull().sum()

    if missing.sum() == 0:
        print("No missing values found ")
    else:
        print(missing[missing > 0])

    print(f"\nTotal Missing Values in Dataset: {df.isnull().sum().sum():,}")

# run for both datasets
dataset_overview(df_full, "SECTION 1: FULL AGRICULTURAL DATASET")
dataset_overview(df_train, "SECTION 2: TRAINING DATASET")

# **HANDLE MISSING VALUES IN TEMPORAL FEATURE (days_diff)**

In [ ]:
# rows containing missing values
missing_rows = df_train[df_train['days_diff'].isnull()]

print("Number of rows with missing days_diff:", len(missing_rows))
print("\nSample rows:")
print(missing_rows[['Grid_ID', 'date', 'days_diff']].head(10))

In [ ]:
# =============================================================================
# Fill first record of each grid with 0 because no previous date exists
# =============================================================================

# check missing values before filling
print("Missing values before fill:", df_train['days_diff'].isnull().sum())

# fill missing values with 0
df_train['days_diff'] = df_train['days_diff'].fillna(0)

# verify after filling
print("Missing values after fill:", df_train['days_diff'].isnull().sum())

# display sample rows
display(df_train[['Grid_ID', 'date', 'days_diff']].head(10))

In [ ]:
# =============================================================================
# CHECK ALL MISSING VALUES IN THE ENTIRE DATASET
# =============================================================================

missing_all = df_train.isnull().sum()

print("Missing values in all columns:")
print(missing_all[missing_all > 0])

print("\nTotal missing values in dataset:", df_train.isnull().sum().sum())

In [ ]:
rows_with_missing = df_train[df_train.isnull().any(axis=1)]

print("Rows containing any missing values:", len(rows_with_missing))
display(rows_with_missing.head(10))

In [ ]:
# =============================================================================
# DATASET OVERVIEW + MISSING VALUES CHECK
# =============================================================================

def dataset_overview(df, dataset_name):
    print("\n" + "="*80)
    print(dataset_name)
    print("="*80)

    print(f"Total Rows: {len(df):,}")
    print(f"Total Columns: {df.shape[1]}")
    print(f"Total Unique Grids: {df['Grid_ID'].nunique():,}")
    print(f"Total Unique Dates: {df['date'].nunique():,}")
    print(f"Start Date: {df['date'].min()}")
    print(f"End Date:   {df['date'].max()}")

    avg_records = df['Grid_ID'].value_counts().mean()
    print(f"Average Records per Grid: {avg_records:.2f}")

    print("\nMissing Values Per Column:")
    missing = df.isnull().sum()

    if missing.sum() == 0:
        print("No missing values found ")
    else:
        print(missing[missing > 0])

    print(f"\nTotal Missing Values in Dataset: {df.isnull().sum().sum():,}")

# run for both datasets
dataset_overview(df_full, "SECTION 1: FULL AGRICULTURAL DATASET")
dataset_overview(df_train, "SECTION 2: TRAINING DATASET")

In [ ]:
# =============================================================================
# SAVE UPDATED TRAINING DATASET (OVERWRITE OLD FILE)
# =============================================================================

df_train.to_csv(training_dataset_path, index=False)

print("Updated file saved successfully and replaced old file ")
print("Saved path:", training_dataset_path)

In [ ]:
df_check = pd.read_csv(training_dataset_path)

print(df_check['days_diff'].isnull().sum())
print(df_check.shape)

In [ ]:
# =============================================================================
# DISPLAY COLUMN NAMES + TOTAL COUNT
# =============================================================================

print("Total Number of Columns:", len(df_train.columns))
print("\nColumn Names:")
print("=" * 50)

for i, col in enumerate(df_train.columns, start=1):
    print(f"{i}. {col}")

# **Feature Correlation Analysis with Stress Label**
Top 20 strongest numerical features affecting future stress prediction

In [ ]:
# =============================================================================
# TOP 20 STRONGEST NUMERICAL FEATURE CORRELATIONS WITH TARGET
# =============================================================================

# keep only numerical columns
numeric_df = df_train.select_dtypes(include=['number'])

# calculate correlation with target
corr_with_target = numeric_df.corr()['stress_label']

# sort by absolute strength
corr_abs = corr_with_target.abs().sort_values(ascending=False)

# display top 20
top20_features = corr_abs.head(20)

print("Top 20 strongest numerical correlations with stress_label:")
print("=" * 70)
print(top20_features)

# actual signed correlations
print("\nActual correlation values:")
print("=" * 70)
print(corr_with_target.loc[top20_features.index])

In [ ]:
# =============================================================================
# VISUALIZE TOP 20 FEATURE CORRELATIONS
# =============================================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 7))

corr_with_target.loc[top20_features.index].sort_values().plot(kind='barh')

plt.title("Top 20 Feature Correlations with Stress Label")
plt.xlabel("Correlation Value")
plt.ylabel("Features")
plt.axvline(x=0)
plt.tight_layout()
plt.show()

# **Categorical Feature Analysis**
Relationship between season, risk label, and future stress

In [ ]:
print(df_train.select_dtypes(include=['object']).columns)

In [ ]:
# =============================================================================
# season + risk_label
# =============================================================================

import matplotlib.pyplot as plt

# -----------------------------
# Season Analysis
# -----------------------------
print("=" * 70)
print("STRESS RATIO BY SEASON")
print("=" * 70)

season_analysis = df_train.groupby('season')['stress_label'].agg(['mean', 'count'])
print(season_analysis)

# plot season
plt.figure(figsize=(6,4))
df_train.groupby('season')['stress_label'].mean().plot(kind='bar')
plt.title("Average Stress Label by Season")
plt.ylabel("Mean Stress Label")
plt.xlabel("Season")
plt.xticks(rotation=0)
plt.show()

# -----------------------------
# Risk Label Analysis
# -----------------------------
print("\n" + "=" * 70)
print("STRESS RATIO BY RISK LABEL")
print("=" * 70)

risk_analysis = df_train.groupby('risk_label')['stress_label'].agg(['mean', 'count'])
print(risk_analysis)

# plot risk label
plt.figure(figsize=(6,4))
df_train.groupby('risk_label')['stress_label'].mean().plot(kind='bar')
plt.title("Average Stress Label by Risk Label")
plt.ylabel("Mean Stress Label")
plt.xlabel("Risk Label")
plt.xticks(rotation=0)
plt.show()

# **Improved Stress Label Analysis**




In [ ]:
# Change path if your file is inside a folder
full_dataset_path = '/content/drive/MyDrive/DEPI PROJECT/agricultural_full_dataset.csv'

df_full = pd.read_csv(full_dataset_path)

print("Full dataset shape:", df_full.shape)
print("Unique grids:", df_full['Grid_ID'].nunique())
display(df_full.head(3))

In [ ]:
# =============================
#  Create working copy and sort time sequence
# =============================

df_model = df_full.copy()

# Make sure date is datetime
df_model['date'] = pd.to_datetime(df_model['date'])

# Sort by grid and time
df_model = df_model.sort_values(['Grid_ID', 'date']).reset_index(drop=True)

unique_grids = df_model['Grid_ID'].nunique()

print("Working dataset shape:", df_model.shape)
print("Unique grids:", unique_grids)
display(df_model[['Grid_ID', 'date']].head(10))

In [ ]:
# =============================
#  Create NDVI_future before deletion
# =============================

df_model['NDVI_future'] = (
    df_model.groupby('Grid_ID')['NDVI']
    .shift(-1)
)

display(df_model[['Grid_ID', 'date', 'NDVI', 'NDVI_future']].head(10))

missing_future = df_model['NDVI_future'].isna().sum()

print("Missing values in NDVI_future:", missing_future)
print("Unique grids:", unique_grids)

if missing_future == unique_grids:
    print("Perfect: one missing future value per grid.")
else:
    print("Warning: mismatch detected.")

In [ ]:
# =============================
#  Verify missing future rows are last rows only
# =============================

missing_rows = df_model[df_model['NDVI_future'].isna()].copy()

last_dates = (
    df_model.groupby('Grid_ID')['date']
    .max()
    .reset_index()
)

check_last = missing_rows.merge(
    last_dates,
    on=['Grid_ID', 'date'],
    how='inner'
)

print("Missing rows count:", missing_rows.shape[0])
print("Matched last-date rows:", check_last.shape[0])

if missing_rows.shape[0] == check_last.shape[0]:
    print("Perfect: all missing future rows are last rows only.")
else:
    print("Warning: some missing rows are not last rows.")

In [ ]:
# =============================
#  Create baseline label (old version)
# =============================

stress_threshold = 0.3

df_model['stress_label_basic'] = (
    df_model['NDVI_future'] < stress_threshold
).astype(int)

print("Baseline label created successfully.")
print(df_model['stress_label_basic'].value_counts())

In [ ]:
# =============================
#  Build season-aware thresholds
# =============================

# Use only valid rows that have NDVI_future
valid_future = df_model[df_model['NDVI_future'].notna()].copy()

# Season-aware thresholds
season_stats = (
    valid_future.groupby('season')
    .agg(
        season_ndvi_future_mean=('NDVI_future', 'mean'),
        season_ndvi_future_std=('NDVI_future', 'std'),
        season_ndvi_future_q25=('NDVI_future', lambda x: x.quantile(0.25)),
        season_veg_health_q25=('veg_health', lambda x: x.quantile(0.25)),
        season_evi_change_q25=('EVI_change', lambda x: x.quantile(0.25))
    )
    .reset_index()
)

display(season_stats)

In [ ]:
# =============================
#  Merge seasonal thresholds back
# =============================

df_model = df_model.merge(season_stats, on='season', how='left')

display(df_model[[
    'season',
    'NDVI_future',
    'season_ndvi_future_q25',
    'veg_health',
    'season_veg_health_q25',
    'EVI_change',
    'season_evi_change_q25'
]].head(10))

In [ ]:
# =============================
#  Create support signals for improved label
# =============================

# Global high drought threshold
drought_q75 = valid_future['drought_severity'].quantile(0.75)

print("Drought severity 75th percentile:", drought_q75)

# Core condition 1: future NDVI is low in absolute terms
df_model['cond_future_low_absolute'] = (df_model['NDVI_future'] < stress_threshold).astype(int)

# Core condition 2: future NDVI is low relative to its season
df_model['cond_future_low_seasonal'] = (
    df_model['NDVI_future'] <= df_model['season_ndvi_future_q25']
).astype(int)

# Core condition 3: future NDVI is lower than current NDVI (actual decline)
df_model['cond_future_decline'] = (
    df_model['NDVI_future'] < df_model['NDVI']
).astype(int)

# Vegetation confirmation
df_model['cond_veg_confirm'] = (
    (df_model['veg_health'] <= df_model['season_veg_health_q25']) |
    (df_model['EVI_change'] <= df_model['season_evi_change_q25'])
).astype(int)

# Environmental support flags
df_model['cond_water_stress'] = (df_model['water_stress'] == 1).astype(int)
df_model['cond_heat_stress']  = (df_model['heat_stress'] == 1).astype(int)
df_model['cond_dry_stress']   = (df_model['dry_stress'] == 1).astype(int)
df_model['cond_vpd_stress']   = (df_model['VPD_stress'] == 1).astype(int)
df_model['cond_drought_high'] = (df_model['drought_severity'] >= drought_q75).astype(int)

# Count environmental supports
df_model['env_support_count'] = (
    df_model['cond_water_stress'] +
    df_model['cond_heat_stress'] +
    df_model['cond_dry_stress'] +
    df_model['cond_vpd_stress'] +
    df_model['cond_drought_high']
)

display(df_model[[
    'Grid_ID', 'date', 'season', 'NDVI', 'NDVI_future',
    'cond_future_low_absolute', 'cond_future_low_seasonal', 'cond_future_decline',
    'cond_veg_confirm', 'env_support_count'
]].head(10))

In [ ]:
# =============================
#  Create improved future stress label
# =============================

df_model['stress_label_improved'] = (
    (df_model['cond_future_low_absolute'] == 1) &
    (df_model['cond_future_low_seasonal'] == 1) &
    (df_model['cond_future_decline'] == 1) &
    (
        (df_model['cond_veg_confirm'] == 1) |
        (df_model['env_support_count'] >= 2)
    )
).astype(int)

print("Improved stress label created successfully.")
print(df_model['stress_label_improved'].value_counts())

In [ ]:
# =============================
#  Remove last row from each grid after label creation
# =============================

rows_before = df_model.shape[0]

df_model = df_model.dropna(subset=['NDVI_future']).reset_index(drop=True)

rows_after = df_model.shape[0]
deleted_rows = rows_before - rows_after

print("Rows before deletion:", rows_before)
print("Rows after deletion:", rows_after)
print("Deleted rows:", deleted_rows)
print("Expected deleted rows:", unique_grids)

if deleted_rows == unique_grids:
    print("Perfect: exactly one row deleted from each grid.")
else:
    print("Warning: mismatch detected.")

In [ ]:
# =============================
#  Compare baseline vs improved label distribution
# =============================

print("Baseline label distribution:")
print(df_model['stress_label_basic'].value_counts())
print("\nImproved label distribution:")
print(df_model['stress_label_improved'].value_counts())

comparison_counts = pd.DataFrame({
    'Baseline_Count': df_model['stress_label_basic'].value_counts(),
    'Improved_Count': df_model['stress_label_improved'].value_counts()
}).fillna(0).astype(int)

display(comparison_counts)

In [ ]:
# =============================
#  Compare agreement between old and improved labels
# =============================

comparison_table = pd.crosstab(
    df_model['stress_label_basic'],
    df_model['stress_label_improved'],
    rownames=['Basic Label'],
    colnames=['Improved Label']
)

display(comparison_table)

agreement_rate = (df_model['stress_label_basic'] == df_model['stress_label_improved']).mean() * 100
change_rate = 100 - agreement_rate

print(f"Agreement rate: {agreement_rate:.2f}%")
print(f"Changed rows rate: {change_rate:.2f}%")

In [ ]:
# =============================
#  Validate risk score vs improved label
# =============================

print("Risk score statistics by improved label:")
display(
    df_model.groupby('stress_label_improved')['risk_score']
    .describe()
)

mean_scores_improved = (
    df_model.groupby('stress_label_improved')['risk_score']
    .mean()
)

print("\nMean risk score by improved label:")
print(mean_scores_improved)

mean_scores_improved.plot(kind='bar', figsize=(6,4))
plt.title("Average Risk Score by Improved Future Stress Label")
plt.xlabel("Improved Future Stress Label")
plt.ylabel("Average Risk Score")
plt.xticks([0,1], ['Healthy (0)', 'Stressed (1)'], rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# =============================
#  Validate improved stress rate by risk label
# =============================

stress_rate_by_risk_improved = (
    df_model.groupby('risk_label')['stress_label_improved']
    .mean()
    .reindex(['Low', 'Medium', 'High']) * 100
)

print(stress_rate_by_risk_improved)

stress_rate_by_risk_improved.plot(kind='bar', figsize=(7,4))
plt.title("Improved Future Stress Rate by Risk Label")
plt.xlabel("Risk Label")
plt.ylabel("Stress Rate (%)")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

# **Validate improved risk score against future stress**

This validation confirms that the **improved risk score** remains strongly aligned with future crop stress patterns.

The future stress rate still increases consistently across the risk levels:

- **Low Risk → 3.88%**
- **Medium Risk → 7.07%**
- **High Risk → 11.59%**

Although the overall stress percentages are lower than the previous version, this is considered an improvement rather than an issue.

The reduction in stress rates indicates that the improved feature engineering process has reduced overestimation and produced more realistic stress probabilities.

Most importantly, the monotonic increasing trend is preserved:

**Low < Medium < High**

This confirms that the risk score remains logically and statistically valid as a predictive feature for future crop stress.

In [ ]:
# =============================
#  Validate improved stress rate by season
# =============================

stress_rate_by_season_improved = (
    df_model.groupby('season')['stress_label_improved']
    .agg(['mean', 'count'])
)

print(stress_rate_by_season_improved)

stress_rate_by_season_improved['mean'].plot(kind='bar', figsize=(7,4))
plt.title("Average Improved Stress Label by Season")
plt.xlabel("Season")
plt.ylabel("Mean Improved Stress Label")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# =============================
#  Compare correlations of both labels with key features
# =============================

key_features = [
    'NDVI_future', 'NDVI', 'NDWI', 'EVI', 'veg_health',
    'water_stress', 'heat_stress', 'dry_stress',
    'VPD_stress', 'drought_severity', 'risk_score'
]

corr_basic = df_model[key_features + ['stress_label_basic']].corr()['stress_label_basic'].drop('stress_label_basic')
corr_improved = df_model[key_features + ['stress_label_improved']].corr()['stress_label_improved'].drop('stress_label_improved')

corr_compare = pd.DataFrame({
    'Basic_Label_Corr': corr_basic,
    'Improved_Label_Corr': corr_improved
}).sort_values('Improved_Label_Corr', key=np.abs, ascending=False)

display(corr_compare)

In [ ]:
# =============================
# FINALIZE TARGET LABEL
# =============================

# Replace final target with improved version
df_model['stress_label'] = df_model['stress_label_improved']

# Delete old comparison columns
df_model = df_model.drop(columns=[
    'stress_label_basic',
    'stress_label_improved'
])

# Check final label
print("Final stress label distribution:")
print(df_model['stress_label'].value_counts())

print("\nFinal dataset shape:")
print(df_model.shape)

In [ ]:
# =============================
# REMOVE TEMPORARY ENGINEERING COLUMNS
# =============================

temp_cols = [
    'cond_future_low_absolute',
    'cond_future_low_seasonal',
    'cond_future_decline',
    'cond_veg_confirm',
    'cond_water_stress',
    'cond_heat_stress',
    'cond_dry_stress',
    'cond_vpd_stress',
    'cond_drought_high',
    'env_support_count',
    'season_ndvi_future_mean',
    'season_ndvi_future_std',
    'season_ndvi_future_q25',
    'season_veg_health_q25',
    'season_evi_change_q25'
]

# Keep only columns that exist
temp_cols = [col for col in temp_cols if col in df_model.columns]

df_model = df_model.drop(columns=temp_cols)

print("Temporary columns removed successfully.")
print("Final dataset shape:", df_model.shape)

In [ ]:
# =============================
# RECREATE DAYS_DIFF FOR VALIDATION
# =============================

df_model['days_diff'] = (
    df_model.groupby('Grid_ID')['date']
    .diff()
    .dt.days
)

print(df_model['days_diff'].describe())
print("\nMissing in days_diff:", df_model['days_diff'].isnull().sum())
print("\nUnique intervals:")
print(df_model['days_diff'].value_counts(dropna=False).head(10))

In [ ]:
# ==============================
# Fill missing values in days_diff
# ==============================

df_model['days_diff'] = df_model['days_diff'].fillna(0)

print(df_model['days_diff'].describe())
print("\nMissing in days_diff:", df_model['days_diff'].isnull().sum())
print("\nUnique intervals:")
print(df_model['days_diff'].value_counts(dropna=False).head(10))

In [ ]:
# ==========================================================
#  DATASET OVERVIEW + COLUMN NAMES
# ==========================================================

print("=" * 80)
print("TRAINING DATASET")
print("=" * 80)

# Basic info
print(f"Total Rows: {df_model.shape[0]:,}")
print(f"Total Columns: {df_model.shape[1]:,}")

# Unique values
print(f"Total Unique Grids: {df_model['Grid_ID'].nunique():,}")
print(f"Total Unique Dates: {df_model['date'].nunique():,}")

# Date range
print(f"Start Date: {df_model['date'].min()}")
print(f"End Date: {df_model['date'].max()}")

# Average records per grid
avg_records = df_model.groupby('Grid_ID').size().mean()
print(f"Average Records per Grid: {avg_records:.2f}")

# Missing values
missing_cols = df_model.isnull().sum()
missing_cols = missing_cols[missing_cols > 0]

print("\nMissing Values Per Column:")
if len(missing_cols) > 0:
    print(missing_cols)
else:
    print("No missing values found.")

print(f"\nTotal Missing Values in Dataset: {df_model.isnull().sum().sum():,}")

# ==========================================================
# COLUMN NAMES
# ==========================================================

print("\n" + "=" * 80)
print("COLUMN NAMES")
print("=" * 80)

for i, col in enumerate(df_model.columns, 1):
    print(f"{i}. {col}")

In [ ]:
# =============================
# FIX days_diff CORRECTLY
# =============================

df_model['days_diff'] = (
    df_model.groupby('Grid_ID')['date']
    .diff()
    .dt.days
)

print(df_model['days_diff'].describe())
print("\nMissing in days_diff:", df_model['days_diff'].isnull().sum())
print("\nUnique intervals:")
print(df_model['days_diff'].value_counts(dropna=False))

In [ ]:
# ==========================================================
#  DATASET OVERVIEW + COLUMN NAMES
# ==========================================================

print("=" * 80)
print("TRAINING DATASET")
print("=" * 80)

# Basic info
print(f"Total Rows: {df_model.shape[0]:,}")
print(f"Total Columns: {df_model.shape[1]:,}")

# Unique values
print(f"Total Unique Grids: {df_model['Grid_ID'].nunique():,}")
print(f"Total Unique Dates: {df_model['date'].nunique():,}")

# Date range
print(f"Start Date: {df_model['date'].min()}")
print(f"End Date: {df_model['date'].max()}")

# Average records per grid
avg_records = df_model.groupby('Grid_ID').size().mean()
print(f"Average Records per Grid: {avg_records:.2f}")

# Missing values
missing_cols = df_model.isnull().sum()
missing_cols = missing_cols[missing_cols > 0]

print("\nMissing Values Per Column:")
if len(missing_cols) > 0:
    print(missing_cols)
else:
    print("No missing values found.")

print(f"\nTotal Missing Values in Dataset: {df_model.isnull().sum().sum():,}")

# ==========================================================
# COLUMN NAMES
# ==========================================================

print("\n" + "=" * 80)
print("COLUMN NAMES")
print("=" * 80)

for i, col in enumerate(df_model.columns, 1):
    print(f"{i}. {col}")

In [ ]:
# =============================================================================
# TOP 20 STRONGEST NUMERICAL FEATURE CORRELATIONS WITH TARGET
# =============================================================================

# keep only numerical columns
numeric_df = df_model.select_dtypes(include=['number'])

# calculate correlation with target
corr_with_target = numeric_df.corr()['stress_label']

# sort by absolute strength
corr_abs = corr_with_target.abs().sort_values(ascending=False)

# save top 20 for plotting
top20_features = corr_abs.head(20)

# display top 20
print("Top 20 strongest numerical correlations with stress_label:")
print("=" * 70)
print(top20_features)

# actual signed correlations
print("\nActual correlation values:")
print("=" * 70)
print(corr_with_target.loc[top20_features.index])

In [ ]:
# =============================================================================
# VISUALIZE TOP 20 FEATURE CORRELATIONS
# hide stress_label only in display
# =============================================================================

import matplotlib.pyplot as plt

# remove target only from plotting
plot_features = top20_features.index.drop('stress_label', errors='ignore')

plt.figure(figsize=(10, 7))

corr_with_target.loc[plot_features].sort_values().plot(kind='barh')

plt.title("Top 20 Feature Correlations with Stress Label")
plt.xlabel("Correlation Value")
plt.ylabel("Features")
plt.axvline(x=0, color='black', linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SEASON + RISK_LABEL
# =============================================================================

import matplotlib.pyplot as plt

# -----------------------------
# Season Analysis
# -----------------------------
print("=" * 70)
print("STRESS RATIO BY SEASON")
print("=" * 70)

season_analysis = df_model.groupby('season')['stress_label'].agg(['mean', 'count'])
print(season_analysis)

# plot season
plt.figure(figsize=(6,4))
df_model.groupby('season')['stress_label'].mean().plot(kind='bar')
plt.title("Average Stress Label by Season")
plt.ylabel("Mean Stress Label")
plt.xlabel("Season")
plt.xticks(rotation=0)
plt.show()

# -----------------------------
# Risk Label Analysis
# -----------------------------
print("\n" + "=" * 70)
print("STRESS RATIO BY RISK LABEL")
print("=" * 70)

risk_analysis = (
    df_model.groupby('risk_label')['stress_label']
    .agg(['mean', 'count'])
    .reindex(['Low', 'Medium', 'High'])
)

print(risk_analysis)

# plot risk label
plt.figure(figsize=(6,4))
df_model.groupby('risk_label')['stress_label'].mean().reindex(['Low', 'Medium', 'High']).plot(kind='bar')
plt.title("Average Stress Label by Risk Label")
plt.ylabel("Mean Stress Label")
plt.xlabel("Risk Label")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# =============================
# SAVE FINAL DATASET INSIDE DEPI PROJECT FOLDER
# =============================

save_path = '/content/drive/MyDrive/DEPI PROJECT/agricultural_training_dataset_improved_final.csv'

# Save file
df_model.to_csv(save_path, index=False)

print("File saved successfully inside DEPI PROJECT folder ")
print("Saved path:")
print(save_path)

In [ ]:
# File path
file_path = '/content/drive/MyDrive/DEPI PROJECT/agricultural_training_dataset_improved_final.csv'

# Load dataset
df = pd.read_csv(file_path)

# Quick check
print("Dataset loaded successfully!")
print("="*50)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# **Exploratory Data Analysis (EDA), Data Validation, and Feature Insights**

In [ ]:
# =============================================================================
# Stress Label Distribution (Class Imbalance Check)
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(x='stress_label', data=df)
plt.title("Stress Label Distribution")
plt.xlabel("Stress Label")
plt.ylabel("Count")
plt.show()

print(df['stress_label'].value_counts(normalize=True))

### **Stress Label Distribution**

The dataset shows a significant class imbalance, with approximately 93% of the samples labeled as non-stress and only around 6.5% labeled as stress.

This imbalance is expected in real-world agricultural data, where stress events occur less frequently compared to normal conditions.

However, this imbalance may affect model performance, as the model could become biased toward the majority class. Therefore, appropriate techniques such as class weighting or resampling should be considered during model training.

In [ ]:
# =============================================================================
#  Correlation Analysis with stress_label
# =============================================================================

import pandas as pd
import matplotlib.pyplot as plt

# keep only numerical columns
numeric_df = df.select_dtypes(include=['number']).copy()

# remove columns that should not be included in correlation analysis
cols_to_drop = ['stress_label', 'NDVI_future']
existing_cols_to_drop = [col for col in cols_to_drop if col in numeric_df.columns]
numeric_df = numeric_df.drop(columns=existing_cols_to_drop)

# remove constant columns to avoid runtime warnings
numeric_df = numeric_df.loc[:, numeric_df.nunique() > 1]

# calculate signed correlation with target
corr_signed = numeric_df.corrwith(df['stress_label'])

# remove NaN correlations if any
corr_signed = corr_signed.dropna().sort_values(ascending=False)

# top 20 strongest features by absolute correlation
top_20_signed = corr_signed.reindex(corr_signed.abs().sort_values(ascending=False).index).head(20)

# print results
print("Top 20 strongest features correlated with stress_label (signed):")
print("=" * 70)
print(top_20_signed)

print("\nTop 10 positive correlations:")
print("=" * 70)
print(corr_signed.head(10))

print("\nTop 10 negative correlations:")
print("=" * 70)
print(corr_signed.tail(10))

# plot
plt.figure(figsize=(10, 7))
top_20_signed.sort_values().plot(kind='barh')
plt.title("Top 20 Features Correlated with Stress Label (Clean Signed)")
plt.xlabel("Correlation")
plt.ylabel("Feature")
plt.show()

### **Feature Correlation with Stress Label**

The correlation analysis reveals clear and meaningful relationships between the features and the stress label.

Vegetation-related features such as NDVI, NDWI, and EVI show strong negative correlations, indicating that healthier vegetation is associated with lower stress levels.

On the other hand, environmental and stress-related features such as temperature-based risk indicators, anomaly flags, and climate stress metrics exhibit positive correlations, suggesting that adverse environmental conditions increase the likelihood of stress.

Overall, these results confirm that agricultural stress is influenced by both vegetation health and environmental factors, validating the effectiveness of the engineered features.

In [ ]:
# =============================================================================
# Top 10 Features Correlation Heatmap
# =============================================================================

# select top 10 correlated features (absolute)
top_10_features = corr_signed.abs().sort_values(ascending=False).head(10).index

# plot
plt.figure(figsize=(8,6))
sns.heatmap(
    df[top_10_features].corr(),
    cmap='coolwarm',
    annot=True,
    fmt=".2f"
)

plt.title("Top 10 Features Correlation Heatmap")
plt.show()

### **Top 10 Features Correlation Heatmap**

The heatmap reveals strong correlations among vegetation-related features such as NDVI, NDWI, EVI, and NDVI_class, indicating that these variables capture similar aspects of vegetation health.

Additionally, change-based features (NDVI_change, EVI_change, NDWI_change, and NDVI_trend) form another correlated group, reflecting temporal dynamics in vegetation conditions.

This suggests the presence of redundancy among features, highlighting the need for feature selection to avoid multicollinearity and improve model efficiency.

In [ ]:
# =============================================================================
# Top 20 Features Correlation Heatmap
# =============================================================================

# select top 20 correlated features (absolute)
top_20_features = corr_signed.abs().sort_values(ascending=False).head(20).index

# plot
plt.figure(figsize=(12,10))
sns.heatmap(
    df[top_20_features].corr(),
    cmap='coolwarm',
    annot=True,
    fmt=".2f"
)

plt.title("Top 20 Features Correlation Heatmap")
plt.show()

### **Top 20 Features Correlation Heatmap**

The extended heatmap provides a broader view of feature relationships, revealing multiple clusters of correlated variables.

Vegetation indices remain highly correlated, while temporal features (lags and changes) also show strong interdependence. Environmental and stress-related features, such as temperature-based metrics and VPD, exhibit distinct correlation patterns, often showing negative relationships with vegetation health indicators.

These observations confirm that agricultural stress is influenced by both vegetation condition and environmental factors, while also indicating potential redundancy among features that should be addressed during feature selection.

In [ ]:
# =============================================================================
# Absolute Correlation Strength Only
# =============================================================================

corr_abs = corr_signed.abs().sort_values(ascending=False)

print("Top 20 strongest features by absolute correlation:")
print("=" * 70)
print(corr_abs.head(20))

In [ ]:
# =============================================================================
# Vegetation Indices Over Time (Mean Trend)
# =============================================================================

df['date'] = pd.to_datetime(df['date'])

time_trend = df.groupby('date')[['NDVI', 'NDWI', 'EVI']].mean()

plt.figure()
time_trend.plot()
plt.title("Vegetation Indices Over Time")
plt.xlabel("Date")
plt.ylabel("Mean Value")
plt.show()

### **Vegetation Indices Over Time**

The vegetation indices (NDVI, EVI, and NDWI) exhibit clear seasonal patterns over time, with periodic increases and decreases reflecting crop growth cycles.

NDVI and EVI show very similar trends, as both measure vegetation health, while NDWI remains lower as it represents water content.

The consistent cyclic behavior indicates that vegetation dynamics are strongly influenced by seasonal factors, confirming the importance of incorporating temporal and seasonal features in the modeling process.

In [ ]:
# =============================================================================
# Climate Variables Over Time
# =============================================================================

climate_trend = df.groupby('date')[['temperature', 'rainfall', 'humidity']].mean()

plt.figure()
climate_trend.plot()
plt.title("Climate Variables Over Time")
plt.xlabel("Date")
plt.ylabel("Mean Value")
plt.show()

### **Climate Variables Over Time**

The climate variables exhibit clear seasonal patterns over time. Temperature follows a periodic trend, increasing during warmer periods and decreasing during cooler seasons.

Humidity shows fluctuations that sometimes appear inversely related to temperature, but this behavior is primarily driven by seasonal changes rather than a consistent direct relationship.

Rainfall remains low for most periods, with occasional spikes, reflecting the climatic characteristics of the region.

Overall, the observed trends highlight the strong influence of seasonality on climate variables.

In [ ]:
# =============================================================================
# NDVI vs NDWI Relationship
# =============================================================================

plt.figure()
sns.scatterplot(x='NDVI', y='NDWI', data=df, alpha=0.3)
plt.title("NDVI vs NDWI")
plt.show()

In [ ]:
# =============================================================================
# NDVI vs EVI Relationship
# =============================================================================

plt.figure()
sns.scatterplot(x='NDVI', y='EVI', data=df, alpha=0.3)
plt.title("NDVI vs EVI")
plt.show()

In [ ]:
# =============================================================================
# Temperature vs NDVI
# =============================================================================

plt.figure()
sns.scatterplot(x='temperature', y='NDVI', data=df, alpha=0.3)
plt.title("Temperature vs NDVI")
plt.show()

In [ ]:
# =============================================================================
# Risk Score vs Stress Label (Validation)
# =============================================================================

plt.figure()
sns.boxplot(x='stress_label', y='risk_score', data=df)
plt.title("Risk Score Distribution by Stress Label")
plt.show()

In [ ]:
# =============================================================================
# Risk Score Bins vs Stress Probability
# =============================================================================

df['risk_bin'] = pd.qcut(df['risk_score'], q=5)

risk_analysis = df.groupby('risk_bin', observed=True)['stress_label'].mean()

plt.figure()
risk_analysis.plot(kind='bar')
plt.title("Stress Probability Across Risk Score Bins")
plt.ylabel("Probability of Stress")
plt.show()

print(risk_analysis)

In [ ]:
# =============================================================================
# Days Difference Validation (Should be ~15 days)
# =============================================================================

plt.figure()
df['days_diff'].dropna().hist(bins=30)
plt.title("Distribution of Days Difference")
plt.xlabel("Days")
plt.ylabel("Frequency")
plt.show()

print(df['days_diff'].value_counts().head(10))

In [ ]:
# =============================================================================
# Outlier Detection (Boxplots)
# =============================================================================

cols = ['NDVI', 'EVI', 'temperature', 'rainfall']

for col in cols:
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f"{col} Distribution")
    plt.show()

### **Distribution Analysis Insights**

The NDVI distribution shows that most values fall within the moderate vegetation range, indicating generally healthy vegetation across the dataset. Some lower values and outliers are observed, which may correspond to areas with weak vegetation, bare soil, or stressed crops.

The EVI distribution appears wider compared to NDVI, reflecting its higher sensitivity to vegetation density and canopy structure. The presence of extreme values suggests that EVI captures more variation in vegetation conditions, including both dense and weak vegetation areas.

The temperature values are distributed within a realistic range, consistent with typical environmental conditions in the studied region. This indicates that the climate data is reliable and suitable for modeling agricultural stress patterns.

The rainfall distribution is highly skewed, with most values concentrated near zero and a few higher values. This pattern is expected in arid and semi-arid regions, where rainfall events are infrequent but can occasionally be intense.

In [ ]:
# =============================================================================
# Stress Distribution Across Seasons
# =============================================================================

plt.figure()
sns.countplot(x='season', hue='stress_label', data=df)
plt.title("Stress Label Distribution Across Seasons")
plt.show()

### **Stress Distribution Across Seasons**

The distribution of stress cases varies across seasons, with higher occurrences observed during Spring and Autumn compared to Winter.

This pattern is expected, as these seasons often involve active crop growth and environmental transitions, which can increase plant sensitivity to stress factors such as temperature fluctuations, water availability, and climate variability.

In contrast, Winter shows fewer stress cases, which may be associated with reduced vegetation activity or more stable environmental conditions.

Overall, the seasonal variation confirms that agricultural stress is influenced by temporal and environmental factors, highlighting the importance of incorporating seasonality into the modeling process.

In [ ]:
# =============================================================================
# Vegetation Indices vs Stress Label
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15,5))

# NDVI
plt.subplot(1, 3, 1)
sns.boxplot(x='stress_label', y='NDVI', data=df)
plt.title("NDVI vs Stress")

# NDWI
plt.subplot(1, 3, 2)
sns.boxplot(x='stress_label', y='NDWI', data=df)
plt.title("NDWI vs Stress")

# EVI
plt.subplot(1, 3, 3)
sns.boxplot(x='stress_label', y='EVI', data=df)
plt.title("EVI vs Stress")

plt.tight_layout()
plt.show()

### **Vegetation Indices vs Stress Label**

The boxplots show a clear difference in vegetation indices between stress and non-stress conditions.

For all three indices (NDVI, NDWI, and EVI), non-stress cases exhibit higher median values compared to stress cases. This indicates that healthier vegetation is associated with lower stress levels.

Although some overlap exists between the two classes, the overall separation suggests that these features are effective indicators of agricultural stress and are highly relevant for model training.

In [ ]:
# =============================================================================
# Climate Variables vs Stress Label
# =============================================================================

plt.figure(figsize=(15,5))

# Temperature
plt.subplot(1, 3, 1)
sns.boxplot(x='stress_label', y='temperature', data=df)
plt.title("Temperature vs Stress")

# Rainfall
plt.subplot(1, 3, 2)
sns.boxplot(x='stress_label', y='rainfall', data=df)
plt.title("Rainfall vs Stress")

# Humidity
plt.subplot(1, 3, 3)
sns.boxplot(x='stress_label', y='humidity', data=df)
plt.title("Humidity vs Stress")

plt.tight_layout()
plt.show()

### **Climate Variables vs Stress Label**

The boxplots illustrate the relationship between climate variables and stress conditions.

Higher temperatures are generally associated with increased stress levels, indicating the impact of heat on vegetation. In contrast, humidity tends to be lower in stress cases, suggesting that drier conditions contribute to plant stress.

Rainfall does not show a strong distinction between the two classes, likely due to its low frequency in the dataset.

Overall, these observations confirm that environmental conditions, particularly temperature and humidity, play a significant role in agricultural stress.

In [ ]:
# ==============================
# Feature Distribution per Class
# ==============================

plt.figure(figsize=(6,4))
sns.kdeplot(data=df, x='NDVI', hue='stress_label', fill=True)
plt.title("NDVI Distribution by Stress")
plt.show()

### **NDVI Distribution by Stress Label**

The distribution shows a clear difference in NDVI values between stress and non-stress conditions.

Non-stress cases tend to have higher NDVI values, indicating healthier vegetation, while stress cases are concentrated at lower NDVI ranges.

Although some overlap exists between the two classes, the overall separation suggests that NDVI is a strong and informative feature for distinguishing between stress and non-stress conditions.

In [ ]:
# =============================================================================
# Stress Distribution Across Season and Month
# =============================================================================

plt.figure(figsize=(14,5))

# Season
plt.subplot(1, 2, 1)
sns.barplot(x='season', y='stress_label', data=df)
plt.title("Stress by Season")
plt.xlabel("Season")
plt.ylabel("Stress Rate")

# Month
plt.subplot(1, 2, 2)
sns.barplot(x='month', y='stress_label', data=df)
plt.title("Stress by Month")
plt.xlabel("Month")
plt.ylabel("Stress Rate")

plt.tight_layout()
plt.show()

# **Model**

## **Environment Setup & Data Loading**

In [ ]:
# Define the path to the raw dataset
file_path = '/content/drive/MyDrive/Kafr_ElSheikh_True1km_TimeSeries_2020_2024_Cleaned.csv'

# Load the dataset
df = pd.read_csv(f'Kafr_ElSheikh_True1km_TimeSeries_2020_2024_Cleaned.csv')

# Convert 'date' column to datetime objects for strict temporal ordering
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
print(f"Raw Data Loaded Successfully! Shape: {df.shape}")

## **Advanced Feature Engineering (The Missing Link)**

In [ ]:
# ==========================================
# Advanced Feature Engineering & Proxies
# ==========================================
print("Engineering temporal features and environmental stressors...")

# Sort by Grid_ID and date to ensure correct chronological shifting
df = df.sort_values(['Grid_ID', 'date'])

# --- A. Temporal Plant Dynamics ---
# 1. Future NDVI (Target Base): Shift NDVI by -1 to predict the next reading
df['NDVI_future'] = df.groupby('Grid_ID')['NDVI'].shift(-1)

# 2. EVI Change: Difference between current and previous EVI
df['EVI_change'] = df.groupby('Grid_ID')['EVI'].diff()

# 3. Vegetation Health: A composite index combining NDVI and EVI
df['veg_health'] = (df['NDVI'] + df['EVI']) / 2

# --- B. Environmental Stress Proxies ---
# Formulating basic climate stress flags based on existing indices (NDWI, NDVI)
df['water_stress'] = (df['NDWI'] < 0.0).astype(int)
df['drought_severity'] = 1 - df['NDWI']
df['heat_stress'] = np.where(df['NDVI'] < 0.2, 1, 0)
df['dry_stress'] = df['water_stress']
df['VPD_stress'] = np.where(df['drought_severity'] > 0.6, 1, 0)

# Drop rows with NaN values created by shifting and differencing (first/last dates)
df = df.dropna(subset=['NDVI_future', 'EVI_change']).copy()

print(f"Feature Engineering Complete! New Shape: {df.shape}")

## **Seasonal Labeling (Building The Target)**

In [ ]:
# ==========================================
# Seasonal Stats & Target Formulation (Strict Temporal Cutoff)
# ==========================================

# Extract month for seasonal calculations
df['month'] = df['date'].dt.month

# --- PREVENTING LOOK-AHEAD BIAS ---
# Determine the cutoff index for the Train Set (80% of data)
train_cutoff_index = int(len(df) * 0.8)

# Calculate the 25th percentile (Q25) using ONLY THE PAST (Train Set)
past_data = df.iloc[:train_cutoff_index]

seasonal_stats = past_data.groupby('month').agg({
    'NDVI_future': lambda x: x.quantile(0.25),
    'veg_health': lambda x: x.quantile(0.25),
    'EVI_change': lambda x: x.quantile(0.25)
}).rename(columns={
    'NDVI_future': 'season_ndvi_future_q25',
    'veg_health': 'season_veg_health_q25',
    'EVI_change': 'season_evi_change_q25'
})

# Merge these past baselines back into the main dataframe
df = df.merge(seasonal_stats, on='month', how='left')

# --- C. Formulating Support Signals ---
stress_threshold = 0.3
# Calculate drought threshold based on past data only
drought_q75 = past_data['drought_severity'].quantile(0.75)

df['cond_future_low_absolute'] = (df['NDVI_future'] < stress_threshold).astype(int)
df['cond_future_low_seasonal'] = (df['NDVI_future'] <= df['season_ndvi_future_q25']).astype(int)
df['cond_future_decline'] = (df['NDVI_future'] < df['NDVI']).astype(int)

df['cond_veg_confirm'] = ((df['veg_health'] <= df['season_veg_health_q25']) |
                          (df['EVI_change'] <= df['season_evi_change_q25'])).astype(int)

df['cond_water_stress'] = (df['water_stress'] == 1).astype(int)
df['cond_heat_stress']  = (df['heat_stress'] == 1).astype(int)
df['cond_dry_stress']   = (df['dry_stress'] == 1).astype(int)
df['cond_vpd_stress']   = (df['VPD_stress'] == 1).astype(int)
df['cond_drought_high'] = (df['drought_severity'] >= drought_q75).astype(int)

df['env_support_count'] = (df['cond_water_stress'] + df['cond_heat_stress'] +
                           df['cond_dry_stress'] + df['cond_vpd_stress'] + df['cond_drought_high'])

# --- D. Final Improved Stress Label ---
df['stress_label_improved'] = (
    (df['cond_future_low_absolute'] == 1) &
    (df['cond_future_low_seasonal'] == 1) &
    (df['cond_future_decline'] == 1) &
    ((df['cond_veg_confirm'] == 1) | (df['env_support_count'] >= 2))
).astype(int)

print("Target Label 'stress_label_improved' Built Successfully without Data Leakage!")
print(df['stress_label_improved'].value_counts())

## **Strict Time-Based Split**

In [ ]:
# ==========================================
# Strict Chronological Data Splitting
# ==========================================
# Sort globally by date to prevent data leakage (predicting past from future)
df = df.sort_values('date')

# Define input features and the target variable
features = ['NDVI', 'veg_health', 'EVI_change', 'water_stress',
            'heat_stress', 'dry_stress', 'VPD_stress', 'drought_severity']
target = 'stress_label_improved'

X = df[features]
y = df[target]

# 80% Train (Past) / 20% Test (Future)
split_index = int(len(X) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Time-Series Split Complete! Train size: {len(X_train)}, Test size: {len(X_test)}")

## **Model Training**

In [ ]:
# ==========================================
# Model 1: Logistic Regression (Linear Baseline)
# ==========================================
print("Training Logistic Regression (Baseline)")

# Creating a pipeline to scale data first, preventing data leakage
lr_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
)

# Train the model
lr_pipeline.fit(X_train, y_train)

# Predict using a 0.80 threshold for consistency across models
lr_preds = (lr_pipeline.predict_proba(X_test)[:, 1] >= 0.80).astype(int)

print("\nLogistic Regression Performance:")
print(classification_report(y_test, lr_preds))

In [ ]:
print("Training Fast Random Forest")

# Initialize Random Forest with constraints for faster execution (max_samples=0.8)
best_rf_fast = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)

# Train the model
best_rf_fast.fit(X_train, y_train)

# Predict using the standard threshold
rf_preds = best_rf_fast.predict(X_test)

print("\nRandom Forest Performance:")
print(classification_report(y_test, rf_preds))

In [ ]:
print(" 1. Performing Time-Series Cross-Validation (Checking Stability)")

# Initialize TimeSeriesSplit with 5 splits (Folds)
tscv = TimeSeriesSplit(n_splits=5)
cv_f1_scores = []

# Dynamically calculate scale_pos_weight
scale_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

# Initialize Base XGBoost
best_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)

# Loop through the time-series folds
fold = 1
for train_index, val_index in tscv.split(X_train):
    # Split the temporal folds
    cv_X_train, cv_X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    cv_y_train, cv_y_val = y_train.iloc[train_index], y_train.iloc[val_index]

    # Train on current fold
    best_xgb.fit(cv_X_train, cv_y_train)

    # Validate on next fold using our custom 0.80 threshold
    fold_probs = best_xgb.predict_proba(cv_X_val)[:, 1]
    fold_preds = (fold_probs >= 0.80).astype(int)

    # Calculate fold score
    score = f1_score(cv_y_val, fold_preds)
    cv_f1_scores.append(score)
    print(f"    Fold {fold}: Validation F1-Score = {score:.3f}")
    fold += 1

print(f" Average Validation F1-Score: {np.mean(cv_f1_scores):.3f}")
print("-" * 50)

print(" 2. Training Final Model on Full Train Set...")
# Train the model one last time on the ENTIRE training set
best_xgb.fit(X_train, y_train)

# Predict on the unseen Test Set
xgb_probs = best_xgb.predict_proba(X_test)[:, 1]
xgb_preds = (xgb_probs >= 0.80).astype(int)

# Print the final classification report
print("\n Final XGBoost Performance on Unseen Test Set (Threshold: 0.80):")
print(classification_report(y_test, xgb_preds))

In [ ]:
# ==========================================
# Create Deployment Dataset
# ==========================================

# Predict on the entire dataset
df["stress_probability"] = best_xgb.predict_proba(df[features])[:, 1]

df["prediction"] = best_xgb.predict(df[features])

# Create Risk Label
def get_risk(prob):

    if prob < 0.4:
        return "Low"

    elif prob < 0.7:
        return "Medium"

    else:
        return "High"

df["risk_label"] = df["stress_probability"].apply(get_risk)

# Save
df.to_csv("deployment_dataset.csv", index=False)

print("Deployment dataset saved successfully!")

In [ ]:
# ==========================================
#  Model Stability Visualization (Time-Series CV)
# ==========================================
print(" Generating Time-Series CV Stability Chart...")

# Data extracted from our Time-Series CV results
folds = ['Fold 1\n(Past)', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5\n(Recent)']
val_scores = [0.471, 0.497, 0.565, 0.395, 0.526]
avg_val = np.mean(val_scores)
final_test_score = 0.62  # Final Unseen Test F1-Score

plt.figure(figsize=(10, 6))
sns.set_style("whitegrid")

# 1. Plot the validation scores (The Learning Journey)
plt.plot(folds, val_scores, marker='o', markersize=10, linestyle='-', linewidth=3,
         color='#3498db', label='Validation F1-Score (Expanding Window)')

# Annotate each point with its value
for i, score in enumerate(val_scores):
    plt.annotate(f'{score:.3f}', (folds[i], val_scores[i]),
                 textcoords="offset points", xytext=(0,12), ha='center',
                 fontweight='bold', fontsize=11, color='#2c3e50')

# 2. Add horizontal line for Average Validation
plt.axhline(y=avg_val, color='#e74c3c', linestyle='--', linewidth=2.5,
            label=f'Average Validation ({avg_val:.3f})')

# 3. Add horizontal line for Final Test Score (The Ultimate Goal)
plt.axhline(y=final_test_score, color='#2ecc71', linestyle='-', linewidth=3,
            label=f'Final Test F1-Score ({final_test_score:.3f})')

# Add an arrow pointing to the final test line to emphasize the leap
plt.annotate('Performance Leap\nafter full training!',
             xy=(4.2, final_test_score), xytext=(3, final_test_score + 0.05),
             arrowprops=dict(facecolor='#2ecc71', shrink=0.05, width=2, headwidth=8),
             fontsize=11, fontweight='bold', color='#27ae60', ha='center')

# Aesthetics and Labels
plt.title('Time-Series Cross-Validation Stability vs. Final Performance', fontsize=15, fontweight='bold', pad=20)
plt.xlabel('Validation Folds (Chronological Order)', fontsize=12, fontweight='bold')
plt.ylabel('F1-Score (Stress Class Detection)', fontsize=12, fontweight='bold')
plt.ylim(0.30, 0.75)  # Setting limits to show the gap clearly
plt.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
plt.tight_layout()

# Save and Display
plt.savefig('time_series_cv_stability.png', dpi=300)
plt.show()

In [ ]:
# ==========================================
# Evaluation Visualizations & Export
# ==========================================
# --- A. ROC Curve Comparison ---
plt.figure(figsize=(9, 6))
models = {'XGBoost': best_xgb, 'Random Forest': best_rf_fast, 'Logistic Regression': lr_pipeline}
colors = ['#e74c3c', '#2ecc71', '#3498db']

for (name, model), color in zip(models.items(), colors):
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold', pad=15)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('roc_curve_comparison.png', dpi=300)
plt.show()

# --- B. Confusion Matrices Comparison ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Diagnostics: Confusion Matrices', fontsize=16, fontweight='bold', y=1.05)

preds_list = [xgb_preds, rf_preds, lr_preds]
titles = ['1. XGBoost (Thresh 0.80)', '2. Random Forest', '3. Logistic Regression']
cm_colors = ['Reds', 'Greens', 'Blues']

for i, ax in enumerate(axes):
    sns.heatmap(confusion_matrix(y_test, preds_list[i]), annot=True, fmt='d', cmap=cm_colors[i], ax=ax, cbar=False)
    ax.set_title(titles[i], fontsize=12, fontweight='bold', pad=15)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_xticklabels(['Healthy (0)', 'Stressed (1)'])
    ax.set_yticklabels(['Healthy (0)', 'Stressed (1)'])

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300)
plt.show()

# --- C. Save Final ML Ready Data ---
df.to_csv('/content/drive/MyDrive/final_processed_data_ML_Ready.csv', index=False)
print("\nProcessed Data Saved and Execution Completed!")

In [ ]:
# ==========================================
#  Advanced Model Benchmarking & Visualizations
# ==========================================
# Styling options for professional plots
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

# 0. Preparation: Gather Predictions and Probabilities
model_preds = {'XGBoost': xgb_preds, 'Random Forest': rf_preds, 'Logistic Regression': lr_preds}
model_probas = {
    'XGBoost': best_xgb.predict_proba(X_test)[:, 1],
    'Random Forest': best_rf_fast.predict_proba(X_test)[:, 1],
    'Logistic Regression': lr_pipeline.predict_proba(X_test)[:, 1]
}
colors = ['#e74c3c', '#2ecc71', '#3498db'] # Red, Green, Blue
features_styled = [f.replace('_', ' ').title() for f in features]

# --- Chart A. Confusion Matrices (Normalized by Rows - Actual Classes) ---
# Goal: Visualize where models make mistakes (False Positives vs False Negatives) in percentages.
print(" Plotting Normalized Confusion Matrices...")
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Error Analysis: Normalized Confusion Matrices (Actual Row %)', fontsize=16, fontweight='bold', y=1.08)

for i, (name, preds) in enumerate(model_preds.items()):
    # Normalize by row (actual class) to see recall visually
    cm = confusion_matrix(y_test, preds, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.1%', cmap=cm_colors[i], ax=axes[i], cbar=False, annot_kws={"size": 12, "fontweight": "bold"})
    axes[i].set_title(f'{name}', fontsize=14, fontweight='bold', pad=10)
    axes[i].set_xlabel('Predicted Label', fontsize=11)
    axes[i].set_ylabel('True Label', fontsize=11)
    axes[i].set_xticklabels(['Healthy (0)', 'Stressed (1)'])
    axes[i].set_yticklabels(['Healthy (0)', 'Stressed (1)'])

plt.tight_layout()
plt.savefig('normalized_confusion_matrices.png', dpi=300)
plt.show()


# --- Chart B. Precision-Recall Curve Comparison ---
# Goal: Crucial for imbalanced data. Shows trade-off between Precision and Recall for Class 1.
print(" Plotting Precision-Recall Curves...")
plt.figure(figsize=(9, 6))
plt.title('Precision-Recall Curve Comparison (Focus on Stress Class)', fontsize=14, fontweight='bold', pad=15)

for (name, probs), color in zip(model_probas.items(), colors):
    precision, recall, _ = precision_recall_curve(y_test, probs)
    average_precision = average_precision_score(y_test, probs)
    plt.plot(recall, precision, color=color, lw=2.5, label=f'{name} (AP = {average_precision:.3f})')

plt.xlabel('Recall (Ability to find all stress cases)', fontsize=12)
plt.ylabel('Precision (Ability to be correct when flagging stress)', fontsize=12)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.legend(loc="upper right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pr_curve_comparison.png', dpi=300)
plt.show()


# --- Chart C. Model Performance Metrics Comparison (Bar Chart) ---
# Goal: Direct visualization of key numbers from the classification report for Class 1.
print(" Plotting Metrics Bar Chart...")
# Prepare data for plotting
metrics_list = []
for name, preds in model_preds.items():
    metrics_list.append({
        'Model': name,
        'Precision (Stressed)': precision_score(y_test, preds),
        'Recall (Stressed)': recall_score(y_test, preds),
        'F1-Score (Stressed)': f1_score(y_test, preds)
    })

df_metrics = pd.DataFrame(metrics_list).melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
# Create grouped bar chart
ax = sns.barplot(data=df_metrics, x='Metric', y='Score', hue='Model', palette=['#34495e', '#a3e635', '#3498db'], edgecolor='white')

# Aesthetics
plt.title('Comparison of Key Metrics for Stress Detection (Class 1)', fontsize=14, fontweight='bold', pad=15)
plt.ylim(0, 1.05)
plt.ylabel('Score (0 to 1)', fontsize=12)
plt.xlabel('')
plt.xticks(fontsize=11, fontweight='bold')
plt.legend(loc='upper left', frameon=True, fontsize=10)

# Add score labels on top of bars
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(format(p.get_height(), '.2f'),
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha = 'center', va = 'center',
                    xytext = (0, 9),
                    textcoords = 'offset points',
                    fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('metrics_bar_chart_comparison.png', dpi=300)
plt.show()


# --- Chart D. Standard ROC Curve Comparison ---
# Goal: General model comparison, standard practice.
print("Plotting ROC Curves...")
plt.figure(figsize=(9, 6))
plt.title('ROC Curve Comparison (General Performance)', fontsize=14, fontweight='bold', pad=15)

for (name, probs), color in zip(model_probas.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve_comparison.png', dpi=300)
plt.show()

# --- E. Export Final ML Ready Data ---
df.to_csv('/content/drive/MyDrive/final_processed_data_ML_Ready.csv', index=False)
print("\nProcessed Data Saved and Advanced Visualization Completed Successfully!")

In [ ]:
# ==========================================
# 9. Explainable AI (XAI): Deep Plant Diagnostics
# ==========================================

def predict_and_explain(row_index, X_data, df_original, model, threshold=0.80):
    """
    Predicts the stress state, displays current plant metrics, and explains the reasoning.
    """
    # 1. Extract features for the specific sample
    sample_features = X_data.iloc[[row_index]]
    original_idx = X_data.index[row_index]

    # 2. Extract meta-data and raw plant metrics from the original dataframe
    grid_id = df_original.loc[original_idx, 'Grid_ID']
    date = df_original.loc[original_idx, 'date'].date()

    # Raw plant metrics (Biological Data)
    ndvi = df_original.loc[original_idx, 'NDVI']
    evi = df_original.loc[original_idx, 'EVI']
    ndwi = df_original.loc[original_idx, 'NDWI']

    # Engineered stress indicators (Environmental Data)
    water_stress = sample_features['water_stress'].values[0]
    heat_stress = sample_features['heat_stress'].values[0]
    drought_val = sample_features['drought_severity'].values[0]
    evi_change = sample_features['EVI_change'].values[0]
    veg_health = sample_features['veg_health'].values[0]

    # 3. Calculate AI stress probability
    stress_probability = model.predict_proba(sample_features)[0][1]
    is_stressed = stress_probability >= threshold

    # 4. Print the Deep Diagnostic Report
    print("=" * 75)
    print(f"SMART CROP INSPECTION REPORT | GRID: {grid_id}")
    print(f"Date of Record: {date}")
    print("=" * 75)

    # --- SECTION 1: Actual Plant Data ---
    print("\nCURRENT PLANT HEALTH METRICS (Biological Readings):")
    # Adding intelligent context to the raw numbers
    ndvi_status = "(Low/Warning)" if ndvi < 0.3 else "(Healthy)"
    ndwi_status = "(Dry/Deficient)" if ndwi < 0 else "(Sufficient Moisture)"

    print(f"   • NDVI (Plant Greenness & Vigor) : {ndvi:.3f} {ndvi_status}")
    print(f"   • NDWI (Leaf Water Content)      : {ndwi:.3f} {ndwi_status}")
    print(f"   • EVI (Canopy Structure/Density) : {evi:.3f}")
    print(f"   • Vegetation Health Composite    : {veg_health:.3f}")

    # --- SECTION 2: AI Prediction ---
    print("\nAI PREDICTION:")
    print(f"   • Predicted Stress Probability : {stress_probability * 100:.1f}%")
    if is_stressed:
        print("   • Final Decision: [STRESSED] - Immediate agricultural intervention recommended.")
    else:
        print("   • Final Decision: [HEALTHY] - Crop is growing under stable conditions.")

    # --- SECTION 3: Reasoning Engine ---
    print("\nEXPLAINABILITY ENGINE (Why did the model predict this?):")
    reasons = []

    # Connecting the raw data to the model's logic
    if water_stress == 1 or ndwi < 0:
        reasons.append(f"Water Deficit: NDWI dropped to ({ndwi:.3f}), falling below the safe threshold and triggering a water stress alert.")

    if drought_val > 0.6:
        reasons.append(f"Drought Conditions: The derived drought severity index is critically high at {drought_val:.2f}, indicating a harsh environment.")

    if heat_stress == 1 or ndvi < 0.2:
        reasons.append(f"Severe Vegetation Loss: NDVI is extremely low ({ndvi:.3f}), indicating a massive loss of chlorophyll, likely due to heat damage.")

    if evi_change < 0:
        reasons.append(f"Negative Growth Trend: The plant's canopy structure degraded by ({evi_change:.3f}) compared to the previous reading.")
    elif evi_change > 0 and is_stressed:
        reasons.append(f"Complex Interaction: Despite a slight structural improvement (+{evi_change:.3f} EVI), environmental stressors (water/drought) dominate the prediction.")

    if is_stressed and not reasons:
        reasons.append("Hidden Pattern: XGBoost detected a complex, non-linear combination of marginal stressors leading to a high-risk prediction.")
    elif not is_stressed:
        reasons.append("Optimal Environment: All key bio-indicators (NDVI, NDWI) and environmental proxies demonstrate stability.")

    for r in reasons:
        print(f"    {r}")
    print("=" * 75)

# ---------------------------------------------------------
# Test the function (Change row_index to test different plots)
# ---------------------------------------------------------
print("Running deep sample diagnostic...\n")
predict_and_explain(row_index=4, X_data=X_test, df_original=df, model=best_xgb)

# **Geospatial Pipeline: Linking AI Model Features with Geographic Coordinates**

In this section, the core agricultural time-series training dataset is spatially linked with the geographical grid coordinates (lat and lon) from the True 1km Grid List. This alignment is a foundational step required to transition tabular machine learning features into georeferenced variables, enabling visual spatial analysis and crop stress mapping across Kafr El-Sheikh Governorate.

In [ ]:
grids_list_path = '/content/drive/My Drive/DEPI PROJECT/Kafr_ElSheikh_True1km_Grids_List.csv'
training_dataset_path = '/content/drive/My Drive/DEPI PROJECT/agricultural_training_dataset_improved_final.csv'


print(" Loading 'Grids List' and 'Improved Final Dataset' from Drive...")
try:
    df_grids = pd.read_csv(grids_list_path)
    df_training = pd.read_csv(training_dataset_path)
    print(" Both datasets loaded successfully")
    print(f" Grids List columns: {list(df_grids.columns)}")
    print(f" Training dataset columns: {list(df_training.columns)}")
except Exception as e:
    print(f" Error loading files: {e}")
    print("Please make sure the file names and folder path match exactly with your Drive.")

# **Merging Datasets**

In [ ]:
globals().keys()

In [ ]:
main_data = df_train
kafr_coordinates = grids_df

# 1. Execute Left Join to map coordinates without losing training features
print(" Integrating geospatial coordinates into the primary dataset...")
merged = pd.merge(main_data, kafr_coordinates, on="Grid_ID", how="left")

# 2. Display the top rows to verify column addition
print(" Previewing the aligned dataset structure:")
display(merged.head())

# 3. Print the matrix shape to confirm data integrity
print("\n================ DATASET INTEGRITY CHECK ================")
print(f" Total Rows (Historical Records): {merged.shape[0]:,}")
print(f" Total Columns (Model Features + Spatial Keys): {merged.shape[1]}")
print("=========================================================\n")

# 4. Save the finalized tracking checkpoint
print(" Exporting georeferenced dataset to CSV...")
merged.to_csv("merged.csv", index=False)
print(" 'merged.csv' has been successfully created and saved")

In [ ]:
print("Exporting and saving georeferenced dataset...")

merged.to_csv("merged_for_mapping_final.csv", index=False)

print("=================== SAVE CHECKPOINT ===================")
print("File Saved Successfully")
print("File: merged_for_mapping_final.csv")
print("Purpose: Cleaned & Aligned Dataset Ready for Interactive Map Plotting.")
print("=======================================================")

# **NDVI Distribution Visualization**

In [ ]:
# Use the active 'merged' dataframe from memory instead of reloading
df = merged

# Filter a snapshot for a specific time-step to analyze spatial density
df_snapshot = df[df['date'] == '2020-01-31']

# Initialize the interactive map centered around Kafr El-Sheikh coordinates
m = folium.Map(location=[df_snapshot['lat'].mean(), df_snapshot['lon'].mean()], zoom_start=10, tiles="CartoDB positron")

# Define the continuous color scale matching the NDVI ranges
colormap = LinearColormap(colors=['red', 'yellow', 'green'], vmin=0.0, vmax=0.6)

# Populate the geographic map with georeferenced data points
for index, row in df_snapshot.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=colormap(row['NDVI']),
        fill=True,
        fill_color=colormap(row['NDVI']),
        fill_opacity=0.7,
        popup=f"Grid: {row['Grid_ID']}<br>NDVI: {row['NDVI']}"
    ).add_to(m)

# Add the dynamic legend scale to the map view
colormap.add_to(m)

# Display the interactive map inline inside Google Colab
m

# **Map Analysis & Scientific Interpretation**
Map Explanation:
This interactive spatial map visualizes the geographical distribution of the Normalized Difference Vegetation Index (NDVI) across Kafr El-Sheikh Governorate for the specific snapshot date of January 31, 2020. The map converts numerical satellite measurements into an intuitive color-coded grid overlay (
1 km
 resolution) to assess regional crop health and canopy density.
Key Visual Insights:
Dominant Green Zones (High NDVI
≥0.40
 to
0.60
): The vast majority of the southern and central agricultural lands appear in deep green. This indicates healthy, dense vegetation with high chlorophyll activity, representing optimal crop growth conditions during this winter cycle.
Yellow Transitional Zones (Moderate NDVI
≈0.30
): These points represent areas with moderate vegetation density, which could indicate fields in early growth stages, mixed vegetation, or transitional agricultural borders.
Red and Orange Coastal Zones (Low NDVI
≤0.20
): Clear clustering of red and orange markers is observed at the northern tips, adjacent to Lake Burullus and the Mediterranean coast. This scientifically reflects low vegetation cover due to higher soil salinity, urban boundaries (like Baltim Resort), or brackish water environments where dynamic crop cultivation is restricted.

# **Agricultural Stress Label Visualization**

In [ ]:
# Use the active 'merged' dataframe from memory
df = merged

# Filter a snapshot for the specific date to analyze stress distribution
df_snapshot = df[df['date'] == '2020-01-31']

# Initialize the interactive map centered around Kafr El-Sheikh coordinates
m = folium.Map(location=[df_snapshot['lat'].mean(), df_snapshot['lon'].mean()], zoom_start=10, tiles="CartoDB positron")

# Define a colormap for stress: Green for Low/No Stress (0.0), Red for High Stress (1.0)
colormap = LinearColormap(colors=['green', 'red'], vmin=0.0, vmax=1.0)

# Populate the geographic map with georeferenced stress points
for index, row in df_snapshot.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=colormap(row['stress_label']),
        fill=True,
        fill_color=colormap(row['stress_label']),
        fill_opacity=0.7,
        popup=f"Grid: {row['Grid_ID']}<br>Stress Label: {row['stress_label']}"
    ).add_to(m)

# Add the dynamic legend scale to the map view
colormap.add_to(m)

m

# **Agricultural Stress Mapping & Spatial Risk Interpretation**
Map Explanation:
This interactive spatial map visualizes the geographic distribution of Agricultural Stress Labels (stress_label) across Kafr El-Sheikh Governorate for the specific snapshot date of January 31, 2020. The map utilizes a binary classification scale mapped onto a continuous gradient (
0.0
 to
1.0
) to immediately isolate resilient crop zones from fields actively undergoing severe environmental or climatic stress.
Key Visual Insights:
Dominant Low-Risk Green Zones (Value = 0.0): The overwhelming majority of the agricultural grid cells in the central and southern regions are shaded in solid green. This indicates stable environmental baseline conditions, proper moisture retention, and zero detected crop stress during this specific monitoring period.
High-Risk Red Hotspots (Value = 1.0): Distinct red markers are clustered primarily along the vulnerable northern coastal boundaries (near Baltim Resort and the Rosetta/Rashid branch estuary), with a few isolated pockets appearing inland. These points mark specific coordinates where localized anomalies—such as elevated soil salinity, sudden moisture deficits, or critical drop-offs in vegetation indices—have triggered an active agricultural stress flag.

# **Temporal Range Exploration**

In [ ]:
# 1. Get all unique dates and sort them chronologically
unique_dates = sorted(merged['date'].unique())

# 2. Print summary of the temporal features
print("=================== TEMPORAL DATA SUMMARY ===================")
print(f" Total Unique Dates (Time-steps): {len(unique_dates)}")
print(f" Earliest Recorded Date (Start):  {unique_dates[0]}")
print(f" Latest Recorded Date (End):     {unique_dates[-1]}")
print("=============================================================\n")

# 3. Display the full list of dates in a clean format
print(" Full List of Available Dates in the Dataset:")
for i, d in enumerate(unique_dates, 1):
    print(f"  {i:02d}. {d}")

# **Interactive Date Selection Map**

In [ ]:
# 1. Get all unique dates from the data automatically and sort them
unique_dates = sorted(merged['date'].unique())

# 2. Create the dropdown widget cleanly
date_dropdown = widgets.Dropdown(
    options=unique_dates,
    value=unique_dates[0],
    description='Target Date:',
    style={'description_width': 'initial'}
)

# Create the action button to trigger refresh cleanly
render_button = widgets.Button(
    description='Render Map',
    button_style='success',
    tooltip='Click to refresh the map with your current selection',
    icon='map'
)

# Combine controls into a single layout row
ui_controls = widgets.HBox([date_dropdown, render_button])

# 3. Define the robust map rendering execution function
def on_button_clicked(b):
    # Force-clear the entire cell output to destroy the old map completely
    clear_output(wait=True)

    # Re-display the UI controls at the top so they do not disappear
    display(ui_controls)

    selected_date = date_dropdown.value

    # Filter the dataset for the dynamically selected date
    df_snapshot = merged[merged['date'] == selected_date]

    if df_snapshot.empty:
        print(f"Warning: No data found for date: {selected_date}")
        return

    print("=============================================================")
    print(f"CURRENT MAP VIEW: {selected_date}")
    print("=============================================================")

    # Initialize the map centered on the current snapshot's coordinates
    m = folium.Map(location=[df_snapshot['lat'].mean(), df_snapshot['lon'].mean()], zoom_start=10, tiles="CartoDB positron")

    # Define the stress colormap: Green (0.0) to Red (1.0)
    colormap = LinearColormap(colors=['green', 'red'], vmin=0.0, vmax=1.0)

    # Plot the georeferenced grid points
    for index, row in df_snapshot.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=6,
            color=colormap(row['stress_label']),
            fill=True,
            fill_color=colormap(row['stress_label']),
            fill_opacity=0.7,
            popup=f"Grid: {row['Grid_ID']}<br>Stress Label: {row['stress_label']}"
        ).add_to(m)

    colormap.add_to(m)

    # Render the map layout cleanly
    display(m)

# 4. Bind the button click event to our rendering function
render_button.on_click(on_button_clicked)

# 5. Trigger the initial rendering setup
display(ui_controls)
on_button_clicked(None)

# **Multi-Feature Spatial Map Viewer**

In [ ]:
# 1. Dynamically get all sorted unique dates
unique_dates = sorted(merged['date'].unique())

# 2. Define the curated list of important features for the dropdown
important_columns = {
    'NDVI (Vegetation Index)': 'NDVI',
    'EVI (Enhanced Vegetation)': 'EVI',
    'NDWI (Water Index)': 'NDWI',
    'Temperature': 'temperature',
    'Rainfall': 'rainfall',
    'VPD (Vapor Pressure Deficit)': 'VPD',
    'Stress Label (Target)': 'stress_label',
    'Risk Score': 'risk_score',
    'Water Stress': 'water_stress',
    'Heat Stress': 'heat_stress',
    'Vegetation Health': 'veg_health'
}

# 3. Create the UI Layout elements cleanly
date_dropdown = widgets.Dropdown(
    options=unique_dates,
    value=unique_dates[0],
    description='Date:',
    style={'description_width': 'initial'}
)

column_dropdown = widgets.Dropdown(
    options=important_columns,
    value='stress_label',
    description='Column:',
    style={'description_width': 'initial'}
)

render_button = widgets.Button(
    description='Render Map',
    button_style='success',
    tooltip='Click to refresh the map with your current selection',
    icon='map'
)

# Combine controls into a single layout row
ui_controls = widgets.HBox([date_dropdown, column_dropdown, render_button])

# 4. Define the robust map rendering execution function
def on_button_clicked(b):
    # Force-clear the entire cell output to destroy the old map completely
    clear_output(wait=True)

    # Re-display the UI controls at the top so they do not disappear
    display(ui_controls)

    selected_date = date_dropdown.value
    selected_column = column_dropdown.value

    # Filter data for the specific date
    df_snapshot = merged[merged['date'] == selected_date]

    if df_snapshot.empty:
        print(f"Warning: No data found for date: {selected_date}")
        return

    # Get dynamic min and max values for the selected column to auto-adjust colors
    min_val = float(df_snapshot[selected_column].min())
    max_val = float(df_snapshot[selected_column].max())

    # Prevent division by zero if all values are identical
    if min_val == max_val:
        max_val += 0.01

    print("=============================================================")
    print(f"DATE: {selected_date}  |  VISUALIZING: {selected_column}")
    print(f"Value Range in Layer: [{min_val:.2f} to {max_val:.2f}]")
    print("=============================================================")

    # Determine color palette based on feature type
    if selected_column in ['stress_label', 'risk_score', 'water_stress', 'heat_stress']:
        colors_palette = ['green', 'yellow', 'red']
    elif selected_column in ['NDVI', 'EVI', 'veg_health']:
        colors_palette = ['red', 'yellow', 'green']
    else:
        colors_palette = ['blue', 'yellow', 'red']

    # Initialize map configuration
    m = folium.Map(location=[df_snapshot['lat'].mean(), df_snapshot['lon'].mean()], zoom_start=10, tiles="CartoDB positron")

    # Build dynamic continuous colormap based on selected column ranges
    colormap = LinearColormap(colors=colors_palette, vmin=min_val, vmax=max_val)

    # Plot georeferenced records
    for index, row in df_snapshot.iterrows():
        current_value = row[selected_column]
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=6,
            color=colormap(current_value),
            fill=True,
            fill_color=colormap(current_value),
            fill_opacity=0.7,
            popup=f"Grid: {row['Grid_ID']}<br>{selected_column}: {current_value:.4f}"
        ).add_to(m)

    colormap.add_to(m)
    display(m)

# 5. Bind the button click event to our rendering function
render_button.on_click(on_button_clicked)

# 6. Trigger the initial rendering setup
display(ui_controls)
on_button_clicked(None)

## **SHAP**

In [ ]:
explainer = shap.TreeExplainer(best_xgb)
sample_size = min(1000, len(X_test))
X_shap = X_test.sample(sample_size, random_state=42)
shap_values = explainer.shap_values(X_shap)

### **SHAP Summary Plot**

In [ ]:
plt.figure(figsize=(10,6))

shap.summary_plot(
    shap_values,
    X_shap,
    show=False
)

plt.tight_layout()
plt.savefig("shap_summary.png", dpi=300)
plt.show()

### **SHAP Bar Plot (Feature Importance)**

In [ ]:
plt.figure(figsize=(8,6))

shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="bar",
    show=False
)

plt.tight_layout()
plt.savefig("shap_feature_importance.png", dpi=300)
plt.show()

### **Waterfall Plot**

In [ ]:
sample_idx = 0

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_idx],
        base_values=explainer.expected_value,
        data=X_shap.iloc[sample_idx],
        feature_names=X_shap.columns
    )
)

# **Save Model**

In [ ]:
SAVE_DIR = "saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(
    list(X_train.columns),
    os.path.join(SAVE_DIR, "feature_names.pkl")
)

print("Saved successfully!")
print(os.listdir(SAVE_DIR))

In [ ]:
model = xgb.XGBClassifier()
model.load_model("saved_models/xgb_model.json")

In [ ]:
import joblib

features = joblib.load("saved_models/feature_names.pkl")
print(features)